In [ ]:
import json
import logging
import sys
import textwrap
import warnings
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.patches import FancyBboxPatch, PathPatch, Rectangle
from matplotlib.path import Path as MplPath

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_DIR = PROJECT_DIR / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from core.config import P, settings  # noqa: E402

settings.PROJECT_DIR = PROJECT_DIR
logging.getLogger("data.annotations").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", category=UserWarning)

FIGURE_DIR = PROJECT_DIR / "research" / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR = PROJECT_DIR / "data" / "processed"
CHECKPOINT_DIR = PROJECT_DIR / "checkpoints"
LOG_DIR = PROJECT_DIR / "logs"

pd.set_option("display.float_format", lambda v: f"{v:,.3f}")

In [ ]:
# --- Estilo ------------------------------------------------------------------
SURFACE = "#ffffff"
INK = "#0b0b0b"
INK_2 = "#525252"
MUTED = "#8a8a8a"
GRID = "#e3e3e3"
BASELINE = "#c4c4c4"

SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]  # slots categóricos 1-4
CRITICAL = "#d03b3b"
WARNING = "#eda100"
GOOD = "#1baf7a"
BLUE = "#2a78d6"
BLUE_FADED = "#b7d3f6"
BLUE_INK = "#0d366b"
PANEL = "#f3f3f3"  # relleno de caja de diagrama

mpl.rcParams.update(
    {
        "figure.facecolor": SURFACE,
        "axes.facecolor": SURFACE,
        "savefig.facecolor": SURFACE,
        "savefig.edgecolor": SURFACE,
        "savefig.dpi": 200,
        "savefig.bbox": "tight",
        "font.family": "sans-serif",
        "font.sans-serif": ["DejaVu Sans"],
        "font.size": 9,
        "text.color": INK,
        "axes.edgecolor": BASELINE,
        "axes.labelcolor": INK_2,
        "axes.titlecolor": INK,
        "axes.titlesize": 10,
        "axes.titleweight": "bold",
        "axes.titlelocation": "left",
        "axes.titlepad": 6,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": False,
        "grid.color": GRID,
        "grid.linewidth": 0.8,
        "xtick.color": MUTED,
        "ytick.color": MUTED,
        "xtick.labelcolor": INK_2,
        "ytick.labelcolor": INK_2,
        "xtick.major.size": 0,
        "ytick.major.size": 0,
        "legend.frameon": False,
        "legend.fontsize": 8.5,
        "lines.linewidth": 2,
    }
)


def style_value_axis(ax, axis="x"):
    """Rejilla fina sólo en el eje de valor; sin espina de línea base en ese eje."""
    ax.grid(True, axis=axis, color=GRID, linewidth=0.8, zorder=0)
    ax.set_axisbelow(True)
    if axis == "x":
        ax.spines["bottom"].set_visible(False)
        ax.spines["left"].set_color(BASELINE)
    else:
        ax.spines["left"].set_visible(False)
        ax.spines["bottom"].set_color(BASELINE)


_K = 0.5523  # constante de control del arco circular en una Bezier cúbica


def _square_path(x0, y0, w, h):
    return MplPath(
        [(x0, y0), (x0 + w, y0), (x0 + w, y0 + h), (x0, y0 + h), (x0, y0)],
        [MplPath.MOVETO] + [MplPath.LINETO] * 3 + [MplPath.CLOSEPOLY],
    )


def _rounded_path(x0, y0, w, h, rx, ry, side="right"):
    """Rectángulo con las dos esquinas de `side` redondeadas: el extremo de dato."""
    rx, ry = min(abs(rx), abs(w)), min(abs(ry), abs(h) / 2)
    if rx <= 0 or ry <= 0 or w == 0 or h == 0:
        return _square_path(x0, y0, w, h)
    x1, y1 = x0 + w, y0 + h

    if side == "right":
        verts = [
            (x0, y0),
            (x1 - rx, y0),
            (x1 - rx + rx * _K, y0),
            (x1, y0 + ry - ry * _K),
            (x1, y0 + ry),
            (x1, y1 - ry),
            (x1, y1 - ry + ry * _K),
            (x1 - rx + rx * _K, y1),
            (x1 - rx, y1),
            (x0, y1),
            (x0, y0),
        ]
    else:  # "top"
        verts = [
            (x0, y0),
            (x1, y0),
            (x1, y1 - ry),
            (x1, y1 - ry + ry * _K),
            (x1 - rx + rx * _K, y1),
            (x1 - rx, y1),
            (x0 + rx, y1),
            (x0 + rx - rx * _K, y1),
            (x0, y1 - ry + ry * _K),
            (x0, y1 - ry),
            (x0, y0),
        ]
    codes = [
        MplPath.MOVETO,
        MplPath.LINETO,
        MplPath.CURVE4,
        MplPath.CURVE4,
        MplPath.CURVE4,
        MplPath.LINETO,
        MplPath.CURVE4,
        MplPath.CURVE4,
        MplPath.CURVE4,
        MplPath.LINETO,
        MplPath.CLOSEPOLY,
    ]
    return MplPath(verts, codes)


def _x_per_y(ax):
    """Unidades de x que se dibujan tan largas como una unidad de y."""
    box = ax.get_position()
    width_in = ax.figure.get_figwidth() * box.width
    height_in = ax.figure.get_figheight() * box.height
    x_range = abs(np.diff(ax.get_xlim())[0])
    y_range = abs(np.diff(ax.get_ylim())[0])
    return (x_range / width_in) / (y_range / height_in)


def barh(ax, ys, widths, color, height=0.62, left=None, radius_frac=0.30, **kw):
    """Barras horizontales con el extremo de dato redondeado. `left` apila."""
    ys = np.atleast_1d(np.asarray(ys, dtype=float))
    widths = np.asarray(widths, dtype=float)
    left = np.zeros_like(widths) if left is None else np.asarray(left, dtype=float)
    ry = height * radius_frac
    rx = ry * _x_per_y(ax)
    for y, w, x0 in zip(ys, widths, left):
        ax.add_patch(
            PathPatch(
                _rounded_path(x0, y - height / 2, w, height, rx, ry, "right"),
                facecolor=color,
                edgecolor="none",
                **kw,
            )
        )
    return ax


def barv(ax, xs, heights, color, width=0.68, radius_frac=0.30, bottom=None, **kw):
    """Barras verticales con la punta redondeada."""
    xs = np.atleast_1d(np.asarray(xs, dtype=float))
    heights = np.asarray(heights, dtype=float)
    bottom = np.zeros_like(heights) if bottom is None else np.asarray(bottom, dtype=float)
    rx = width * radius_frac
    ry = rx / _x_per_y(ax)
    for x, h, y0 in zip(xs, heights, bottom):
        ax.add_patch(
            PathPatch(
                _rounded_path(x - width / 2, y0, width, h, rx, ry, "top"),
                facecolor=color,
                edgecolor="none",
                **kw,
            )
        )
    return ax


# --- Salida ------------------------------------------------------------------
# Dos artefactos por figura: el PNG que `main.tex` incluye hoy y, cuando la figura es
# una serie de datos y no un diagrama, el JSON con esos datos, para poder redibujarla
# en pgfplots sin volver a correr el cuaderno.
DATA_DIR = FIGURE_DIR / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)


def _jsonable(value):
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, (pd.Series, pd.Index)):
        return value.tolist()
    if isinstance(value, pd.DataFrame):
        return value.to_dict(orient="list")
    raise TypeError(type(value))


def save(fig, name, data=None):
    path = FIGURE_DIR / f"{name}.png"
    fig.savefig(path)
    if data is not None:
        target = DATA_DIR / f"{name}.json"
        target.write_text(json.dumps(data, indent=1, ensure_ascii=False, default=_jsonable))

In [ ]:
# --- Helpers de diagrama ------------------------------------------------------


def canvas(width_in=10.0, height_in=5.5):
    fig, ax = plt.subplots(figsize=(width_in, height_in))
    ax.set_xlim(0, 100)
    ax.set_ylim(0, 100 * height_in / width_in)
    ax.set_aspect("equal")
    ax.axis("off")
    return fig, ax


def wrap_text(text, width=None):
    """Ajusta a `width` respetando los saltos de línea que ya trae el texto."""
    text = str(text)
    if not width:
        return text
    return "\n".join(
        textwrap.fill(line, width) if line.strip() else line for line in text.split("\n")
    )


def _units_per_point(ax):
    """Unidades del lienzo que mide un punto tipográfico (lienzo de 100 de ancho)."""
    return 100.0 / (ax.figure.get_figwidth() * 72.0)


def _stack(ax, blocks):
    """Alturas y separaciones de una pila de bloques de texto, en unidades del lienzo."""
    upt = _units_per_point(ax)
    heights = [(text.count("\n") + 1) * size * 1.34 * upt for text, size, _, _ in blocks]
    gaps = [1.15 * (blocks[i][1] + blocks[i + 1][1]) / 2 * upt for i in range(len(blocks) - 1)]
    return heights, gaps, sum(heights) + sum(gaps)


def _blocks(
    title, body, foot, fs, body_fs, foot_fs, ink, body_ink, foot_ink, weight, wrap, body_wrap
):
    blocks = []
    if title:
        blocks.append((wrap_text(title, wrap), fs, ink, weight))
    if body:
        blocks.append((wrap_text(body, body_wrap or wrap or 30), body_fs, body_ink, "normal"))
    if foot:
        blocks.append((wrap_text(foot, body_wrap or wrap or 30), foot_fs, foot_ink, "normal"))
    return blocks


def node(
    ax,
    cx,
    cy,
    w,
    h=None,
    title="",
    body=None,
    foot=None,
    *,
    fill=PANEL,
    edge=BASELINE,
    ink=INK,
    body_ink=INK_2,
    foot_ink=MUTED,
    lw=1.1,
    fs=9.0,
    body_fs=7.8,
    foot_fs=7.0,
    weight="bold",
    dashed=False,
    wrap=None,
    body_wrap=None,
    radius=1.4,
    pad_pt=9.0,
    zorder=2,
):
    """Caja redondeada centrada en (cx, cy) con hasta tres bloques de texto apilados.

    Con `h=None` la caja se ajusta al texto: es lo que evita las cajas medio vacías y
    los títulos encimados cuando el contenido cambia de largo.
    """
    blocks = _blocks(
        title, body, foot, fs, body_fs, foot_fs, ink, body_ink, foot_ink, weight, wrap, body_wrap
    )
    heights, gaps, total = _stack(ax, blocks)
    if h is None:
        h = total + 2 * pad_pt * _units_per_point(ax)

    ax.add_patch(
        FancyBboxPatch(
            (cx - w / 2, cy - h / 2),
            w,
            h,
            boxstyle=f"round,pad=0,rounding_size={radius}",
            linewidth=lw,
            edgecolor=edge,
            facecolor=fill,
            linestyle=(0, (4, 3)) if dashed else "-",
            zorder=zorder,
            mutation_aspect=1,
        )
    )

    y = cy + total / 2
    for i, (text, size, color, wt) in enumerate(blocks):
        ax.text(
            cx,
            y - heights[i] / 2,
            text,
            ha="center",
            va="center",
            fontsize=size,
            color=color,
            fontweight=wt,
            zorder=zorder + 1,
            linespacing=1.32,
        )
        y -= heights[i] + (gaps[i] if i < len(gaps) else 0)
    return cx, cy, w, h


def node_height(
    ax,
    title,
    body=None,
    foot=None,
    *,
    fs=9.0,
    body_fs=7.8,
    foot_fs=7.0,
    wrap=None,
    body_wrap=None,
    pad_pt=9.0,
):
    blocks = _blocks(
        title, body, foot, fs, body_fs, foot_fs, INK, INK_2, MUTED, "bold", wrap, body_wrap
    )
    return _stack(ax, blocks)[2] + 2 * pad_pt * _units_per_point(ax)


def flow_row(ax, items, cy, *, margin=2.5, gap=2.6, h=None, arrow_color=BLUE, arrows=True, **kw):
    """Fila de cajas que reparte el ancho del lienzo, con flechas entre ellas.

    `items` es una lista de dicts con `title` y, opcionalmente, `body`, `foot` y `kw`.
    El alto es común a toda la fila: el de la caja que más texto lleva.
    """
    n = len(items)
    w = (100 - 2 * margin - (n - 1) * gap) / n
    if h is None:
        h = max(
            node_height(
                ax,
                item["title"],
                item.get("body"),
                item.get("foot"),
                fs=kw.get("fs", 9.0),
                body_fs=kw.get("body_fs", 7.8),
                foot_fs=kw.get("foot_fs", 7.0),
                wrap=item.get("kw", {}).get("wrap", kw.get("wrap")),
                body_wrap=kw.get("body_wrap"),
            )
            for item in items
        )
    boxes = []
    for i, item in enumerate(items):
        cx = margin + w / 2 + i * (w + gap)
        options = dict(kw)
        options.update(item.get("kw", {}))
        boxes.append(
            node(ax, cx, cy, w, h, item["title"], item.get("body"), item.get("foot"), **options)
        )
        if arrows and i:
            arrow(ax, (cx - w / 2 - gap, cy), (cx - w / 2, cy), color=arrow_color, lw=1.3)
    return boxes


def arrow(ax, start, end, *, color=MUTED, lw=1.2, rad=0.0, dashed=False, head="-|>", zorder=1):
    ax.annotate(
        "",
        xy=end,
        xytext=start,
        zorder=zorder,
        arrowprops=dict(
            arrowstyle=head,
            color=color,
            linewidth=lw,
            shrinkA=1.5,
            shrinkB=1.5,
            linestyle=(0, (4, 3)) if dashed else "-",
            connectionstyle=f"arc3,rad={rad}",
            mutation_scale=11,
        ),
    )


def elbow(ax, points, *, color=BLUE, lw=1.3):
    """Conector en ángulo recto con punta de flecha en el último tramo."""
    xs, ys = zip(*points)
    ax.plot(xs, ys, color=color, linewidth=lw, solid_capstyle="round", zorder=1)
    arrow(ax, points[-2], points[-1], color=color, lw=lw)


def caption_at(ax, x, y, text, *, fs=7.8, color=INK_2, ha="center", va="center", wrap=None, **kw):
    ax.text(
        x,
        y,
        wrap_text(text, wrap),
        ha=ha,
        va=va,
        fontsize=fs,
        color=color,
        linespacing=1.35,
        **kw,
    )


def chip(ax, cx, cy, text, color=BLUE, *, fs=7.5, pad=1.0, ink=SURFACE):
    """Etiqueta pequeña de fondo lleno: para marcar una salida o un estado."""
    ax.text(
        cx,
        cy,
        text,
        ha="center",
        va="center",
        fontsize=fs,
        color=ink,
        fontweight="bold",
        zorder=5,
        bbox=dict(boxstyle=f"round,pad={pad * 0.3}", facecolor=color, edgecolor="none"),
    )


# --- Glifos de tensor ---------------------------------------------------------
# `node` dibuja una caja con texto: sirve para un módulo, no para el dato que viaja
# entre módulos. Esto es el vocabulario que faltaba -- el espectrograma como plano, los
# tokens contables, la pirámide a escala -- para que un diagrama muestre la forma del
# tensor en vez de describirla por escrito.


def slab(ax, cx, cy, w, h, *, skew=0.16, fill=SURFACE, edge=INK, lw=1.2, zorder=2):
    """Plano en perspectiva: el espectrograma y los mapas de salida."""
    dx = w * skew
    points = [
        (cx - w / 2 + dx, cy - h / 2),
        (cx + w / 2, cy - h / 2),
        (cx + w / 2 - dx, cy + h / 2),
        (cx - w / 2, cy + h / 2),
    ]
    ax.add_patch(
        plt.Polygon(points, closed=True, facecolor=fill, edgecolor=edge, lw=lw, zorder=zorder)
    )
    return points


def token_strip(
    ax, cx, cy, w, h, n, *, vertical=True, fill=SURFACE, edge=INK, lw=0.9, gap=0.30, zorder=3
):
    """Tokens dibujados de a uno: se ven y se cuentan, no se leen en un pie de caja."""
    if vertical:
        cell = h / n
        for k in range(n):
            ax.add_patch(
                Rectangle(
                    (cx - w / 2, cy - h / 2 + k * cell + gap / 2),
                    w,
                    cell - gap,
                    facecolor=fill,
                    edgecolor=edge,
                    lw=lw,
                    zorder=zorder,
                )
            )
    else:
        cell = w / n
        for k in range(n):
            ax.add_patch(
                Rectangle(
                    (cx - w / 2 + k * cell + gap / 2, cy - h / 2),
                    cell - gap,
                    h,
                    facecolor=fill,
                    edgecolor=edge,
                    lw=lw,
                    zorder=zorder,
                )
            )


def feature_pyramid(
    ax, cx, cy, w, h, levels, *, fill=SURFACE, edge=INK, lw=1.1, label_y=None, zorder=3
):
    """Un plano por nivel, con el alto proporcional a su resolución.

    La proporción entre los niveles *es* el dato: dibujarlos todos iguales borra lo
    único que distingue una pirámide multiescala de tres convoluciones en fila.
    """
    tallest = max(rows for rows, _ in levels)
    slot = w / len(levels)
    for k, (rows, cols) in enumerate(levels):
        x = cx - w / 2 + slot * (k + 0.5)
        height = h * (rows / tallest) ** 0.55
        ax.add_patch(
            Rectangle(
                (x - slot * 0.22, cy - height / 2),
                slot * 0.44,
                height,
                facecolor=fill,
                edgecolor=edge,
                lw=lw,
                zorder=zorder,
            )
        )
        y = cy - h / 2 - 1.4 if label_y is None else label_y
        caption_at(ax, x, y, f"{rows}x{cols}", fs=6.0, color=MUTED, va="top")


def patch_grid(ax, cx, cy, w, h, n_cols, n_rows, *, edge=INK, lw=0.9, zorder=3):
    """La rejilla de parches sobre la entrada: el parcheo del AST hecho visible."""
    ax.add_patch(
        Rectangle(
            (cx - w / 2, cy - h / 2),
            w,
            h,
            facecolor=SURFACE,
            edgecolor=edge,
            lw=lw,
            zorder=zorder,
        )
    )
    for i in range(1, n_cols):
        x = cx - w / 2 + w * i / n_cols
        ax.plot([x, x], [cy - h / 2, cy + h / 2], color=edge, lw=lw * 0.7, zorder=zorder + 1)
    for j in range(1, n_rows):
        y = cy - h / 2 + h * j / n_rows
        ax.plot([cx - w / 2, cx + w / 2], [y, y], color=edge, lw=lw * 0.7, zorder=zorder + 1)


def detection_slab(ax, cx, cy, w, h, boxes, *, color=CRITICAL, **kw):
    """El plano de salida con las cajas encima: dónde termina el pipeline."""
    slab(ax, cx, cy, w, h, **kw)
    for bx, by, bw, bh in boxes:
        ax.add_patch(
            Rectangle(
                (cx - w / 2 + (bx - bw / 2) * w, cy - h / 2 + (by - bh / 2) * h),
                bw * w,
                bh * h,
                facecolor="none",
                edgecolor=color,
                lw=1.3,
                zorder=6,
            )
        )


def zoom_arrow(ax, cx, cy, w, h, *, fill=PANEL, edge=MUTED, lw=1.0, zorder=1):
    """Flecha hueca hacia abajo: 'esto de arriba se abre en este panel'."""
    shaft, head = w * 0.42, h * 0.46
    points = [
        (cx - shaft / 2, cy + h / 2),
        (cx + shaft / 2, cy + h / 2),
        (cx + shaft / 2, cy - h / 2 + head),
        (cx + w / 2, cy - h / 2 + head),
        (cx, cy - h / 2),
        (cx - w / 2, cy - h / 2 + head),
        (cx - shaft / 2, cy - h / 2 + head),
    ]
    ax.add_patch(
        plt.Polygon(points, closed=True, facecolor=fill, edgecolor=edge, lw=lw, zorder=zorder)
    )


def detail_panel(ax, cx, cy, w, h, title="", *, edge=BASELINE, lw=1.1, fs=7.6, zorder=0):
    """El recuadro que enmarca un panel de detalle del segundo piso."""
    ax.add_patch(
        FancyBboxPatch(
            (cx - w / 2, cy - h / 2),
            w,
            h,
            boxstyle="round,pad=0,rounding_size=1.6",
            facecolor=SURFACE,
            edgecolor=edge,
            lw=lw,
            zorder=zorder,
        )
    )
    if title:
        caption_at(
            ax,
            cx - w / 2 + 2.0,
            cy + h / 2 - 2.2,
            title,
            fs=fs,
            color=INK,
            ha="left",
            va="top",
            weight="bold",
        )

In [ ]:
import soundfile as sf  # noqa: E402
import torch  # noqa: E402
import torchaudio  # noqa: E402

from utils.audio import hz_to_y, mel_spectrogram, y_to_hz  # noqa: E402

MEL = mel_spectrogram(P)


def load_segment(audio_path, start_s=0.0, duration_s=3.0, params=P):
    """Trozo mono de un `.wav`, remuestreado a `params.target_sr`."""
    with sf.SoundFile(str(audio_path)) as handle:
        handle.seek(int(start_s * handle.samplerate))
        frames = handle.read(int(duration_s * handle.samplerate), dtype="float32", always_2d=True)
        waveform = torch.from_numpy(frames.mean(axis=1))
        if handle.samplerate != params.target_sr:
            waveform = torchaudio.functional.resample(waveform, handle.samplerate, params.target_sr)
    return waveform


def log_mel(waveform, params=P):
    return np.log(MEL(waveform).numpy() + params.eps)


def draw_spectrogram(ax, image, duration_s, *, params=P, cmap="magma", n_ticks=5, vrange=(2, 99.5)):
    """Espectrograma log-mel con el eje de frecuencia en el espacio mel normalizado."""
    lo, hi = np.percentile(image, vrange)
    ax.imshow(
        image,
        origin="lower",
        aspect="auto",
        cmap=cmap,
        extent=(0, duration_s, 0, 1),
        interpolation="nearest",
        vmin=lo,
        vmax=hi,
    )
    ticks = np.linspace(0, 1, n_ticks)
    ax.set_yticks(ticks, [f"{y_to_hz(np.array([t]), params)[0] / 1000:.1f}" for t in ticks])
    ax.set_ylabel("frecuencia (kHz, eje mel)")
    ax.set_xlabel("tiempo (s)")
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    return ax


def draw_box(
    ax,
    t0,
    t1,
    low_hz,
    high_hz,
    *,
    params=P,
    color=GOOD,
    label=None,
    lw=1.5,
    dashed=False,
    fs=7.5,
    label_below=False,
):
    """Caja tiempo–frecuencia sobre un espectrograma dibujado por `draw_spectrogram`."""
    y0 = float(hz_to_y(np.array([max(low_hz, params.f_min)]), params)[0])
    y1 = float(hz_to_y(np.array([min(high_hz, params.f_max)]), params)[0])
    ax.add_patch(
        FancyBboxPatch(
            (t0, y0),
            t1 - t0,
            y1 - y0,
            boxstyle="round,pad=0,rounding_size=0.012",
            linewidth=lw,
            edgecolor=color,
            facecolor="none",
            linestyle=(0, (4, 2)) if dashed else "-",
            zorder=4,
        )
    )
    if label:
        ax.text(
            t0,
            (y0 - 0.022) if label_below else (y1 + 0.014),
            label,
            color=color,
            fontsize=fs,
            fontweight="bold",
            va="top" if label_below else "bottom",
            ha="left",
            zorder=5,
        )
    return y0, y1

In [ ]:
from data.annotations import load_annotations  # noqa: E402
from data.species import CALL_TYPES, Species  # noqa: E402
from prepare_data import (  # noqa: E402
    EXCLUDED_PAIRS,
    JOINED_PAIRS,
    LABEL_BY,
    LABEL_COLUMN,
    MIN_PAIR_COUNT,
)

SOURCE_DIR = settings.data_dir / "cleaned"
annotations = load_annotations(SOURCE_DIR)
annotations["pair"] = annotations.species + "/" + annotations.call_type

SPECIES_NAME = {s.name.lower(): s.value.replace("_", " ") for s in Species}
SPECIES_LATIN = {
    "aa": "Aotus azarae",
    "ac": "Ateles chamek",
    "as": "Alouatta sara",
    "cc": "Cebus cuscinus",
    "lw": "Leontocebus weddelli",
    "pt": "Plecturocebus toppini",
    "sb": "Saimiri boliviensis",
    "sm": "Sapajus macrocephalus",
}
CALL_NAME = {
    f"{s.name.lower()}/{code}": label.replace("_", " ")
    for s, codes in CALL_TYPES.items()
    for code, label in codes.items()
}
CALL_NAME["lw/trino"] = "trino (tr, tj, tt y tf fusionados)"


def select_experiment_from(df):
    """El mismo filtro de `prepare_data.select_experiment`, sin releer el disco."""
    out = df.copy()
    excluded = out[["species", "call_type"]].apply(tuple, axis=1).isin(EXCLUDED_PAIRS)
    out = out[~excluded]
    out["low_freq_hz"] = out["low_freq_hz"].clip(lower=P.f_min)
    for old_pairs, new_pair in JOINED_PAIRS.items():
        for old_pair in old_pairs:
            joined = (out["species"] == old_pair[0]) & (out["call_type"] == old_pair[1])
            out.loc[joined, ["species", "call_type"]] = new_pair
    pairs = out[["species", "call_type"]].apply(tuple, axis=1)
    counts = pairs.value_counts()
    keep = counts[counts >= MIN_PAIR_COUNT].index
    out = out[pairs.isin(keep)].copy()
    out["label"] = LABEL_COLUMN[LABEL_BY](out)
    out["pair"] = out["label"]
    return out


experiment = select_experiment_from(annotations)
CLASSES = sorted(experiment.pair.unique())


def recording_table():
    cache = CACHE_DIR / "recording_durations.csv"
    if cache.exists():
        return pd.read_csv(cache)
    rows = []
    for wav in sorted(SOURCE_DIR.rglob("*.wav")):
        try:
            info = sf.info(str(wav))
        except Exception:
            continue
        folder = next((p.name for p in wav.parents if "__" in p.name), wav.parent.name)
        rows.append(
            {
                "audio_path": str(wav),
                "species": folder.split("__")[-1].lower(),
                "duration_s": info.duration,
                "has_annotation": wav.with_suffix(".txt").exists(),
            }
        )
    table = pd.DataFrame(rows)
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    table.to_csv(cache, index=False)
    return table


recordings = recording_table()

In [ ]:
# --- Elegir ejemplos reales de forma determinista -----------------------------


def pick_annotation(pair, *, df=None, rank=0, source=None):
    """Anotación de `pair` cuya duración es la `rank`-ésima más cercana a la mediana."""
    table = annotations if df is None else df
    rows = table[table.pair == pair]
    if source is not None:
        rows = rows[rows.audio_path.str.contains(source)]
    if rows.empty:
        raise LookupError(f"sin anotaciones para {pair}")
    order = (rows.duration_s - rows.duration_s.median()).abs().sort_values().index
    return rows.loc[order[min(rank, len(order) - 1)]]


def busiest_segment(pair, window_s=6.0, *, df=None, n_events=6):
    """Ventana de `window_s` con un número de eventos de `pair` cercano a `n_events`."""
    table = annotations if df is None else df
    rows = table[table.pair == pair]
    best = []
    for audio_path, group in rows.groupby("audio_path"):
        for start in group.begin_time_s.to_numpy():
            inside = group[(group.begin_time_s >= start) & (group.end_time_s <= start + window_s)]
            if len(inside):
                best.append(
                    (
                        abs(len(inside) - n_events),
                        -float(inside.duration_s.sum()),
                        audio_path,
                        start,
                    )
                )
    best.sort()
    _, _, audio_path, start = best[0]
    inside = table[
        (table.audio_path == audio_path)
        & (table.end_time_s > start)
        & (table.begin_time_s < start + window_s)
    ]
    return audio_path, float(start), inside


def segment_around(row, pad_ratio=0.9, min_pad_s=0.25, params=P):
    """Inicio y duración de un trozo centrado en la anotación `row`."""
    pad = max(row.duration_s * pad_ratio, min_pad_s)
    start = max(row.begin_time_s - pad, 0.0)
    return start, float(row.duration_s + 2 * pad)

In [ ]:
row = pick_annotation("lw/cs")
start_s, duration_s = segment_around(row, pad_ratio=1.6, min_pad_s=0.5)
waveform = load_segment(row.audio_path, start_s, duration_s)

fig, ax = plt.subplots(figsize=(8.2, 4.4))
draw_spectrogram(ax, log_mel(waveform), duration_s)

t0, t1 = row.begin_time_s - start_s, row.end_time_s - start_s
y0, y1 = draw_box(
    ax,
    t0,
    t1,
    row.low_freq_hz,
    row.high_freq_hz,
    color=GOOD,
    label=f"{row.species.upper()} / {row.call_type.upper()}  ·  {CALL_NAME.get(row.pair, '')}",
    lw=1.8,
)

guide = dict(color=SURFACE, linewidth=0.9, linestyle=(0, (3, 3)), zorder=3, alpha=0.85)
for t in (t0, t1):
    ax.plot([t, t], [0, y0], **guide)
for y in (y0, y1):
    ax.plot([t1, duration_s], [y, y], **guide)

for t, name in ((t0, r"$t_{ini}$"), (t1, r"$t_{fin}$")):
    ax.annotate(
        name,
        xy=(t, 0),
        xytext=(0, -16),
        textcoords="offset points",
        ha="center",
        color=INK_2,
        fontsize=9,
    )
for y, name in ((y0, r"$f_{min}$"), (y1, r"$f_{max}$")):
    ax.annotate(
        name,
        xy=(1, y),
        xycoords=("axes fraction", "data"),
        xytext=(6, 0),
        textcoords="offset points",
        va="center",
        ha="left",
        color=INK_2,
        fontsize=9,
        annotation_clip=False,
    )

save(fig, "annotation_example")
plt.show()

In [ ]:
TARGET = "lw/cs"
duration_s = 6.0
audio_path, start_s, inside = busiest_segment(
    TARGET, window_s=duration_s, df=experiment, n_events=6
)
inside = inside[inside.pair == TARGET]
waveform = load_segment(audio_path, start_s, duration_s)
image = log_mel(waveform)
events = [
    (r.begin_time_s - start_s, r.end_time_s - start_s, r.low_freq_hz, r.high_freq_hz, r.pair)
    for r in inside.itertuples()
]

fig, axes = plt.subplots(
    3,
    1,
    figsize=(9.6, 7.6),
    sharex=True,
    gridspec_kw={"height_ratios": [1.0, 0.42, 1.35], "hspace": 0.30},
)

ax = axes[0]
seg = 0.5
edges = np.arange(0, duration_s + seg, seg)
covered = np.array(
    [
        sum(max(0, min(e, hi) - max(s, lo)) for s, e, *_ in events) / seg
        for lo, hi in zip(edges[:-1], edges[1:])
    ]
)
probability = np.clip(covered * 1.6, 0, 1)
ax.step(
    np.repeat(edges, 2)[1:-1],
    np.repeat(probability, 2),
    color=MUTED,
    linewidth=1.8,
    where="pre",
)
ax.fill_between(
    np.repeat(edges, 2)[1:-1], np.repeat(probability, 2), step="pre", color=GRID, alpha=0.9
)
for edge in edges:
    ax.axvline(edge, color=GRID, linewidth=0.7, zorder=0)
ax.set_ylim(0, 1.25)
ax.set_yticks([0, 0.5, 1.0])
ax.set_ylabel("p(llamada)")
ax.set_title("(a)  Probabilidad por segmento fijo", fontsize=10, pad=10)

ax = axes[1]
for t0, t1, *_ in events:
    ax.add_patch(
        FancyBboxPatch(
            (t0, 0.32),
            t1 - t0,
            0.36,
            boxstyle="round,pad=0,rounding_size=0.05",
            facecolor=MUTED,
            edgecolor="none",
        )
    )
ax.set_ylim(0, 1)
ax.set_yticks([])
ax.set_title("(b)  Evento delimitado en el tiempo (onset--offset)", fontsize=10, pad=10)
for side in ("left", "top", "right"):
    ax.spines[side].set_visible(False)

ax = axes[2]
draw_spectrogram(ax, image, duration_s)
for i, (t0, t1, low, high, pair) in enumerate(events):
    draw_box(ax, t0, t1, low, high, color=GOOD, lw=1.5)
chip(ax, duration_s * 0.06, 0.90, f"{len(events)} x {TARGET.upper()}", GOOD, fs=8)
ax.set_title("(c)  Caja tiempo--frecuencia con etiqueta", fontsize=10, pad=10)

save(fig, "output_forms")
plt.show()

In [ ]:
TREE = {
    "problem": (
        "El análisis del audio de monitoreo acústico pasivo no acompaña al ritmo de su "
        "recolección, y las anotaciones que sí se producen no son consistentes entre sí"
    ),
    "causes": [
        ("C1", "El tiempo de analista experto es escaso y no escala con el volumen recolectado"),
        ("C2", "Los anotadores no coinciden entre sí ni consigo mismos"),
        ("C3", "La salida de los sistemas automáticos no sustituye el trabajo del analista"),
    ],
    "effects": [
        (
            "E1",
            "Equipo de investigación: el acervo grabado crece más rápido de lo que puede "
            "convertirse en datos analizables",
        ),
        (
            "E2",
            "Desarrollo metodológico: no hay una referencia confiable contra la cual medir "
            "el avance",
        ),
    ],
    "indirect": (
        "E3",
        "Ecología y gestión: las preguntas se responden tarde o sobre muestras sesgadas",
    ),
    "conditions": [
        (
            "S1",
            "El abaratamiento de los sensores desacopló la capacidad de recolectar de la de analizar",
        ),
        ("S2", "La unidad acústica no tiene una definición estable ni fronteras discretas"),
    ],
}

fig, ax = canvas(12.6, 9.2)
TOP = 100 * 9.2 / 12.6  # 73.0

AXIS_X0, AXIS_X1 = 9, 73
COL = [21, 41, 61]

# --- causas (abajo) ---
for (code, text), cx in zip(TREE["causes"], COL):
    node(ax, cx, 8, 19, 14, code, text, wrap=25, fill=SURFACE, edge=BLUE, lw=1.3, fs=10)
    arrow(ax, (cx, 15.4), (cx if cx == COL[1] else 41, 24.2), color=BLUE, lw=1.3)

# --- problema central ---
node(
    ax,
    41,
    30,
    60,
    11,
    "PROBLEMA CENTRAL",
    TREE["problem"],
    wrap=64,
    fill=BLUE,
    edge=BLUE,
    ink=SURFACE,
    body_ink=SURFACE,
    fs=8.5,
    body_fs=8.6,
    lw=1.4,
)

# --- efectos directos ---
for (code, text), cx in zip(TREE["effects"], (23, 59)):
    arrow(ax, (41, 35.6), (cx, 44.4), color=SERIES[1], lw=1.3)
    node(ax, cx, 51, 30, 13, code, text, wrap=38, fill=SURFACE, edge=SERIES[1], lw=1.3, fs=10)
    arrow(ax, (cx, 57.6), (41, 64.6), color=SERIES[1], lw=1.3, dashed=True)

# --- efecto indirecto ---
code, text = TREE["indirect"]
node(ax, 41, 69, 44, 9.5, code, text, wrap=52, fill=SURFACE, edge=SERIES[1], lw=1.3, fs=10)

# --- condiciones subyacentes, fuera del eje causal ---
caption_at(ax, 88, 62, "CONDICIONES SUBYACENTES", fs=8, color=MUTED, weight="bold")
for (code, text), cy in zip(TREE["conditions"], (44, 24)):
    node(
        ax,
        88,
        cy,
        22,
        16,
        code,
        text,
        wrap=26,
        fill=SURFACE,
        edge=MUTED,
        dashed=True,
        lw=1.1,
        fs=9.5,
    )

for label, cy, color in (
    ("EFECTOS", 60, SERIES[1]),
    ("PROBLEMA", 30, BLUE),
    ("CAUSAS", 8, BLUE),
):
    ax.text(
        2.5,
        cy,
        label,
        rotation=90,
        ha="center",
        va="center",
        fontsize=8,
        color=color,
        fontweight="bold",
    )
ax.plot([6, 6], [1, 17], color=BLUE, linewidth=1.2, alpha=0.35)
ax.plot([6, 6], [24, 36], color=BLUE, linewidth=1.2, alpha=0.35)
ax.plot([6, 6], [44, TOP - 0.5], color=SERIES[1], linewidth=1.2, alpha=0.35)
ax.plot([76.5, 76.5], [14, 66], color=BASELINE, linewidth=1.0, linestyle=(0, (3, 3)))

save(fig, "problem_tree")
plt.show()

In [ ]:
PHASES = [
    ("1. Comprensión de\nla investigación", "Árbol de problemas,\nobjetivos y criterios", "Cap. 1"),
    ("2. Comprensión\nde los datos", "Exploración y calidad\nde las anotaciones", "OE1 · Cap. 4"),
    ("3. Preparación\nde datos", "Curación, vocabulario,\npartición y ventaneo", "OE1 · Cap. 4"),
    ("4. Modelado", "Detector de cajas\ny línea base", "OE2 · Cap. 5"),
    ("5. Evaluación", "Métricas, errores y\ntiempo de revisión", "OE2, OE3 · Cap. 5"),
    ("6. Despliegue", "Inferencia por lotes\ny export a Raven", "OE2 · Cap. 5"),
]

fig, ax = canvas(10.6, 8.8)
TOP = 100 * 8.8 / 10.6
cx0, cy0, radius = 50, TOP / 2, 31.0

angles = np.linspace(90, -270, len(PHASES), endpoint=False)
positions = [
    (cx0 + radius * np.cos(np.radians(a)), cy0 + radius * np.sin(np.radians(a))) for a in angles
]

ring = 15.5
theta = np.linspace(0, 2 * np.pi, 400)
ax.plot(
    cx0 + ring * np.cos(theta),
    cy0 + ring * np.sin(theta),
    color=BASELINE,
    linewidth=1.0,
    linestyle=(0, (4, 4)),
)
for a in angles + 30:
    a0, a1 = np.radians(a + 5), np.radians(a - 5)
    arrow(
        ax,
        (cx0 + ring * np.cos(a0), cy0 + ring * np.sin(a0)),
        (cx0 + ring * np.cos(a1), cy0 + ring * np.sin(a1)),
        color=BASELINE,
        lw=1.6,
    )

for (x, y), (name, content, tag) in zip(positions, PHASES):
    node(
        ax,
        x,
        y,
        26,
        None,
        name,
        content,
        tag,
        wrap=24,
        fill=SURFACE,
        edge=BLUE,
        lw=1.3,
        fs=8.8,
        body_fs=7.4,
        foot_fs=7.2,
        foot_ink=BLUE,
    )

node(
    ax,
    cx0,
    cy0,
    19,
    None,
    "DATOS",
    f"{len(recordings):,} grabaciones\n{len(annotations):,} anotaciones",
    wrap=22,
    fill=PANEL,
    edge=BASELINE,
    fs=9,
    body_fs=7.6,
)

save(fig, "crisp_dm")
plt.show()

In [ ]:
per_species = recordings.groupby("species").agg(
    recorded_h=("duration_s", lambda s: s.sum() / 3600),
    files=("audio_path", "count"),
)
per_species["reviewed_h"] = (
    (recordings[recordings.has_annotation].groupby("species").duration_s.sum() / 3600)
    .reindex(per_species.index)
    .fillna(0)
)
per_species["event_min"] = (
    (annotations.groupby("species").duration_s.sum() / 60).reindex(per_species.index).fillna(0)
)
per_species["share"] = per_species.reviewed_h / per_species.recorded_h
per_species = per_species.sort_values("recorded_h", ascending=False)


order = per_species.index.tolist()
names = [f"{SPECIES_NAME[s]}\n{SPECIES_LATIN.get(s, '')}" for s in order]
ys = np.arange(len(order))

fig, axes = plt.subplots(
    1, 2, figsize=(12.4, 5.6), gridspec_kw={"width_ratios": [1.55, 1], "wspace": 0.42}
)

ax = axes[0]
peak = per_species.recorded_h.max()
ax.set_xlim(0, peak * 1.22)
ax.set_ylim(len(order) - 0.4, -0.9)
barh(ax, ys, per_species.recorded_h.values, BLUE_FADED, height=0.62)
barh(ax, ys, per_species.reviewed_h.values, BLUE, height=0.30)
for y, (_, r) in zip(ys, per_species.iterrows()):
    ax.text(
        r.recorded_h + peak * 0.015,
        y,
        f"{r.recorded_h:,.1f} h · {r.share:.0%} con anotación",
        va="center",
        ha="left",
        fontsize=7.6,
        color=INK_2,
    )
ax.set_yticks(ys, names, fontsize=8)
style_value_axis(ax, "x")
ax.set_xlabel("horas de grabación")

ax = axes[1]
peak = per_species.event_min.max()
ax.set_xlim(0, peak * 1.30)
ax.set_ylim(len(order) - 0.4, -0.9)
barh(ax, ys, per_species.event_min.values, SERIES[1], height=0.5)
for y, value in zip(ys, per_species.event_min.values):
    ax.text(
        value + peak * 0.02,
        y,
        f"{value:,.0f} min",
        va="center",
        ha="left",
        fontsize=7.6,
        color=INK_2,
    )
ax.set_yticks(ys, [""] * len(order))
style_value_axis(ax, "x")
ax.set_xlabel("minutos de evento anotado")

save(
    fig,
    "recorded_vs_annotated",
    data={
        "especie": order,
        "nombre": [SPECIES_NAME[s] for s in order],
        "latin": [SPECIES_LATIN.get(s, "") for s in order],
        "horas_grabadas": per_species.recorded_h,
        "horas_con_anotacion": per_species.reviewed_h,
        "minutos_de_evento": per_species.event_min,
    },
)
plt.show()

In [ ]:
CHAIN = [
    ("Vocalización", "el aparato fonador fija\nla banda de frecuencia", "—"),
    (
        "Propagación",
        "atenuación con la distancia,\nenmascaramiento del coro",
        "pierde SNR, sobre todo en agudos",
    ),
    (
        "Grabadora",
        f"AudioMoth, {P.target_sr / 1000:.1f} kHz,\nmono",
        f"pierde lo que supera {P.f_max / 1000:.1f} kHz",
    ),
    (
        "STFT",
        f"ventana {P.win_length} ({P.win_length / P.target_sr * 1000:.0f} ms)\nsalto {P.hop_length} ({P.hop_length / P.target_sr * 1000:.0f} ms)",
        "compromiso tiempo / frecuencia",
    ),
    (
        "Mel + compresión",
        f"{P.n_mels} bandas HTK\n{P.f_min:.0f} Hz – {P.f_max / 1000:.1f} kHz",
        "resolución no uniforme",
    ),
    ("Caja anotada", "cuatro coordenadas\ny dos etiquetas", "un rectángulo aproxima un contorno"),
]

fig, ax = canvas(13.2, 4.2)
TOP = 100 * 4.2 / 13.2
boxes = flow_row(
    ax,
    [{"title": name, "body": body} for name, body, _ in CHAIN],
    TOP * 0.70,
    margin=1.5,
    gap=1.8,
    wrap=21,
    fill=SURFACE,
    edge=BLUE,
    lw=1.3,
    fs=8.8,
    body_fs=7.2,
)
for (cx, cy, w, h), (_, _, loss) in zip(boxes, CHAIN):
    caption_at(
        ax,
        cx,
        cy - h / 2 - 4.0,
        loss,
        fs=7.0,
        color=CRITICAL if loss != "—" else MUTED,
        wrap=24,
        va="top",
    )
save(fig, "signal_chain")
plt.show()

In [ ]:
row = pick_annotation("lw/cs")
start_s, duration_s = segment_around(row, pad_ratio=1.2, min_pad_s=0.35)
waveform = load_segment(row.audio_path, start_s, duration_s)

fig, axes = plt.subplots(1, 3, figsize=(13.0, 4.3), sharey=True, gridspec_kw={"wspace": 0.10})
for ax, win in zip(axes, (256, P.win_length, 4096)):
    transform = torchaudio.transforms.MelSpectrogram(
        sample_rate=P.target_sr,
        n_fft=P.n_fft,
        win_length=win,
        hop_length=P.hop_length,
        n_mels=P.n_mels,
        f_min=P.f_min,
        f_max=P.f_max,
        power=2.0,
        mel_scale=P.mel_scale,
    )
    image = np.log(transform(waveform).numpy() + P.eps)
    draw_spectrogram(ax, image, duration_s)
    draw_box(
        ax,
        row.begin_time_s - start_s,
        row.end_time_s - start_s,
        row.low_freq_hz,
        row.high_freq_hz,
        color=GOOD,
        lw=1.4,
    )
    chosen = win == P.win_length
    ax.set_title(
        f"win_length = {win}  ·  Δt {win / P.target_sr * 1000:.0f} ms"
        f"  ·  Δf {P.target_sr / win:.0f} Hz",
        fontsize=9.5,
        color=INK if chosen else INK_2,
        pad=10,
    )
    if ax is not axes[0]:
        ax.set_ylabel("")

save(fig, "stft_tradeoff")
plt.show()

In [ ]:
hz = np.linspace(0, P.f_max, 2000)
y = hz_to_y(hz, P)

fig, axes = plt.subplots(
    1, 2, figsize=(12.0, 5.2), gridspec_kw={"width_ratios": [1, 1.15], "wspace": 0.30}
)

ax = axes[0]
ax.plot(hz / 1000, y, color=BLUE, linewidth=2.2)
ax.plot([0, P.f_max / 1000], [0, 1], color=BASELINE, linewidth=1.2, linestyle=(0, (4, 3)))
ax.set_xlim(0, P.f_max / 1000)
ax.set_ylim(0, 1)
half = float(y_to_hz(np.array([0.5]), P)[0])
ax.plot([0, half / 1000], [0.5, 0.5], color=SERIES[1], linewidth=1.2, linestyle=(0, (3, 3)))
ax.plot([half / 1000, half / 1000], [0, 0.5], color=SERIES[1], linewidth=1.2, linestyle=(0, (3, 3)))
ax.annotate(
    f"la mitad del eje\ncabe bajo {half / 1000:.1f} kHz",
    xy=(half / 1000, 0.5),
    xytext=(half / 1000 + 2.0, 0.30),
    color=SERIES[1],
    fontsize=8.5,
    arrowprops=dict(arrowstyle="-", color=SERIES[1], linewidth=1),
)
ax.text(P.f_max / 1000 * 0.70, 0.68, "escala lineal", color=MUTED, fontsize=8, rotation=21)
ax.set_xlabel("frecuencia (kHz)")
ax.set_ylabel("posición en la imagen (eje mel normalizado)")
ax.grid(True, color=GRID, linewidth=0.7)
ax.set_axisbelow(True)

# --- derecha: qué le cuesta a la resolución --------------------------------
ax = axes[1]
edges = y_to_hz(np.linspace(0, 1, P.n_mels + 1), P)
centres = (edges[:-1] + edges[1:]) / 2
widths = np.diff(edges)
ax.plot(centres / 1000, widths, color=BLUE, linewidth=2.2)
ax.set_xlim(0, P.f_max / 1000)
ax.set_ylim(0, widths.max() * 1.18)
for target in (1000.0, 10000.0):
    index = int(np.argmin(np.abs(centres - target)))
    ax.scatter([centres[index] / 1000], [widths[index]], s=30, color=BLUE, zorder=5)
    ax.annotate(
        f"a {centres[index] / 1000:.0f} kHz, cada banda\ncubre {widths[index]:.0f} Hz",
        xy=(centres[index] / 1000, widths[index]),
        xytext=(centres[index] / 1000 + 2.2, widths[index] + widths.max() * 0.12),
        fontsize=8,
        color=INK_2,
        arrowprops=dict(arrowstyle="-", color=MUTED, linewidth=0.9),
    )
for edge in edges[::4]:
    ax.plot([edge / 1000] * 2, [0, widths.max() * 0.035], color=BASELINE, linewidth=0.8)
ax.set_xlabel("frecuencia (kHz)")
ax.set_ylabel("ancho de la banda mel (Hz)")
style_value_axis(ax, "y")

save(
    fig,
    "mel_axis",
    data={
        "hz": hz[::20],
        "y_mel": y[::20],
        "banda_centro_hz": centres,
        "banda_ancho_hz": widths,
    },
)
plt.show()

In [ ]:
from models.pcen import TrainablePCEN  # noqa: E402

sample = experiment.sample(min(24, len(experiment)), random_state=7)
noisiest, best_floor = None, -np.inf
for r in sample.itertuples():
    try:
        clip = load_segment(r.audio_path, max(r.begin_time_s - 1.0, 0.0), P.clip_len_s)
    except Exception:
        continue
    power = MEL(clip)
    floor = float(np.percentile(np.log(power.numpy() + P.eps), 25))
    if floor > best_floor:
        noisiest, best_floor = r, floor

clip = load_segment(noisiest.audio_path, max(noisiest.begin_time_s - 1.0, 0.0), P.clip_len_s)
clip_start = max(noisiest.begin_time_s - 1.0, 0.0)
power = MEL(clip).unsqueeze(0).unsqueeze(0)
with torch.no_grad():
    pcen = TrainablePCEN(n_mels=P.n_mels)(power)[0, 0].numpy()
log_image = np.log(power[0, 0].numpy() + P.eps)

fig, axes = plt.subplots(1, 2, figsize=(12.4, 4.6), sharey=True, gridspec_kw={"wspace": 0.10})
for ax, image, name in ((axes[0], log_image, "log-mel"), (axes[1], pcen, "PCEN")):
    draw_spectrogram(ax, image, P.clip_len_s)
    draw_box(
        ax,
        max(noisiest.begin_time_s - clip_start, 0),
        min(noisiest.end_time_s - clip_start, P.clip_len_s),
        noisiest.low_freq_hz,
        noisiest.high_freq_hz,
        color=GOOD,
        label=noisiest.pair.upper(),
        lw=1.5,
    )
    ax.set_title(name, fontsize=10.5, pad=10)
axes[1].set_ylabel("")

save(fig, "pcen_vs_logmel")
plt.show()

In [ ]:
from data.manifest import _boxes_in_window  # noqa: E402

row = pick_annotation("lw/cs")
clip_start = max(row.begin_time_s - 1.1, 0.0)
group = annotations.loc[[row.name]]
boxes, _, _ = _boxes_in_window(group, np.zeros(1), clip_start, P)
cx, cy, w, h = boxes[0]
waveform = load_segment(row.audio_path, clip_start, P.clip_len_s)

fig = plt.figure(figsize=(12.6, 4.8))
grid = fig.add_gridspec(1, 2, width_ratios=[1.35, 1], wspace=0.26)

ax = fig.add_subplot(grid[0])
draw_spectrogram(ax, log_mel(waveform), P.clip_len_s)
t0, t1 = row.begin_time_s - clip_start, row.end_time_s - clip_start
y0, y1 = draw_box(ax, t0, t1, row.low_freq_hz, row.high_freq_hz, color=GOOD, lw=1.8)
ax.plot([t0, t1], [(y0 + y1) / 2] * 2, color=GOOD, linewidth=1, linestyle=(0, (3, 2)))
ax.plot([(t0 + t1) / 2] * 2, [y0, y1], color=GOOD, linewidth=1, linestyle=(0, (3, 2)))
ax.scatter([(t0 + t1) / 2], [(y0 + y1) / 2], s=26, color=GOOD, zorder=6)

ax = fig.add_subplot(grid[1])
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_aspect("equal")
ax.add_patch(Rectangle((0, 0), 1, 1, facecolor=PANEL, edgecolor=BASELINE, linewidth=1.1))
ax.add_patch(
    Rectangle(
        (cx - w / 2, cy - h / 2), w, h, facecolor=BLUE, alpha=0.18, edgecolor=BLUE, linewidth=1.6
    )
)
ax.scatter([cx], [cy], s=26, color=BLUE, zorder=5)
ax.annotate(
    "",
    xy=(cx - w / 2, cy - h / 2 - 0.06),
    xytext=(cx + w / 2, cy - h / 2 - 0.06),
    arrowprops=dict(arrowstyle="<|-|>", color=BLUE, linewidth=1, mutation_scale=9),
)
ax.text(cx, cy - h / 2 - 0.10, f"w = {w:.3f}", ha="center", va="top", fontsize=8, color=BLUE)
ax.annotate(
    "",
    xy=(cx + w / 2 + 0.06, cy - h / 2),
    xytext=(cx + w / 2 + 0.06, cy + h / 2),
    arrowprops=dict(arrowstyle="<|-|>", color=BLUE, linewidth=1, mutation_scale=9),
)
ax.text(cx + w / 2 + 0.09, cy, f"h = {h:.3f}", ha="left", va="center", fontsize=8, color=BLUE)
ax.annotate(
    f"(cx, cy) = ({cx:.3f}, {cy:.3f})",
    xy=(cx, cy),
    xytext=(0.06, min(cy + 0.22, 0.94)),
    fontsize=8,
    color=BLUE,
    fontweight="bold",
    arrowprops=dict(arrowstyle="-", color=BLUE, linewidth=0.9),
)
ax.grid(True, color=GRID, linewidth=0.7)
ax.set_axisbelow(True)
ax.set_xticks([0, 0.5, 1], ["0", "1.5 s", f"{P.clip_len_s:g} s"])
ax.set_yticks(
    [0, 0.5, 1],
    ["25 Hz", f"{y_to_hz(np.array([0.5]), P)[0] / 1000:.1f} kHz", f"{P.f_max / 1000:.1f} kHz"],
)
ax.set_xlabel("x = t / clip_len_s")
ax.set_ylabel("y = hz_to_y(f)")
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
caption_at(
    ax,
    0.5,
    -0.30,
    f"{row.duration_s * 1000:.0f} ms y {row.bandwidth_hz / 1000:.1f} kHz "
    f"→ ({cx:.3f}, {cy:.3f}, {w:.3f}, {h:.3f})",
    fs=8.5,
    color=INK,
)

save(fig, "box_coordinates")
plt.show()

In [ ]:
from torchvision.ops import box_iou  # noqa: E402

from utils.boxes import _min_area_iou  # noqa: E402

CASES = [
    (
        "Borde desplazado",
        (0.20, 0.30, 0.62, 0.72),
        (0.26, 0.33, 0.68, 0.75),
        "el anotador y el modelo no cierran el evento en el mismo sitio",
    ),
    (
        "Encuadre laxo",
        (0.18, 0.25, 0.70, 0.78),
        (0.24, 0.31, 0.65, 0.72),
        "misma detección, caja más floja: castiga IoU 0,75 y no IoU 0,25",
    ),
    (
        "Sílaba dentro de frase",
        (0.10, 0.20, 0.90, 0.85),
        (0.34, 0.38, 0.52, 0.66),
        "IoU la separa; IoMin la delata como anidada",
    ),
]

fig, axes = plt.subplots(1, 3, figsize=(12.2, 4.6), gridspec_kw={"wspace": 0.16})
for ax, (name, a, b, _note) in zip(axes, CASES):
    box_a = torch.tensor([a], dtype=torch.float32)
    box_b = torch.tensor([b], dtype=torch.float32)
    iou = float(box_iou(box_a, box_b)[0, 0])
    iomin = float(_min_area_iou(torch.cat([box_a, box_b]))[0, 1])

    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_aspect("equal")
    ax.add_patch(Rectangle((0, 0), 1, 1, facecolor=PANEL, edgecolor=BASELINE, linewidth=1.0))
    ax.add_patch(
        Rectangle(
            (a[0], a[1]), a[2] - a[0], a[3] - a[1], facecolor="none", edgecolor=GOOD, linewidth=1.8
        )
    )
    ax.add_patch(
        Rectangle(
            (b[0], b[1]),
            b[2] - b[0],
            b[3] - b[1],
            facecolor="none",
            edgecolor=BLUE,
            linewidth=1.8,
            linestyle=(0, (4, 2)),
        )
    )
    lo = (max(a[0], b[0]), max(a[1], b[1]))
    hi = (min(a[2], b[2]), min(a[3], b[3]))
    ax.add_patch(
        Rectangle(lo, hi[0] - lo[0], hi[1] - lo[1], facecolor=BLUE, alpha=0.16, edgecolor="none")
    )
    ax.set_xticks([])
    ax.set_yticks([])
    for side in ("top", "right", "left", "bottom"):
        ax.spines[side].set_visible(False)
    ax.set_title(name, fontsize=10, pad=22)
    ax.annotate(
        f"IoU {iou:.2f}",
        xy=(0, 1),
        xytext=(0, 4),
        xycoords="axes fraction",
        textcoords="offset points",
        fontsize=9.5,
        color=INK,
        fontweight="bold",
        va="bottom",
    )
    ax.annotate(
        f"IoMin {iomin:.2f}",
        xy=(0.42, 1),
        xytext=(0, 4),
        xycoords="axes fraction",
        textcoords="offset points",
        fontsize=9.5,
        color=INK_2,
        va="bottom",
    )

handles = [
    plt.Line2D([], [], color=GOOD, linewidth=2, label="anotación"),
    plt.Line2D([], [], color=BLUE, linewidth=2, linestyle=(0, (4, 2)), label="predicción"),
]
axes[0].legend(handles=handles, loc="lower left", bbox_to_anchor=(0, -0.32), ncol=2)
save(fig, "iou_criterion")
plt.show()

In [ ]:
BUILD_MANIFEST = True

manifest = splits = None
if BUILD_MANIFEST:
    from data.manifest import build_manifest, split_manifest  # noqa: E402
    from data.species import LabelSet  # noqa: E402

    labels = LabelSet(CLASSES)
    manifest = build_manifest(experiment, labels, empty_ratio=0.25, seed=42)
    splits = dict(
        zip(
            ("train", "val", "test"),
            split_manifest(manifest, seed=42, n_classes=len(labels), ratios=(0.6, 0.225, 0.175)),
        )
    )
    n_boxes = sum(len(w.boxes) for w in manifest)

In [ ]:
n_loaded = len(annotations)
n_pairs = annotations.pair.nunique()
excluded_mask = annotations[["species", "call_type"]].apply(tuple, axis=1).isin(EXCLUDED_PAIRS)
n_no_phrase = int((~excluded_mask).sum())
n_experiment = len(experiment)
n_review = int(annotations.requires_review.sum())

STEPS = [
    (
        "data/cleaned/",
        f"{len(recordings):,} grabaciones\n{recordings.has_annotation.sum():,} con .txt",
        "material recibido",
    ),
    (
        "Carga y consolidación",
        f"{n_loaded:,} anotaciones\n{n_pairs} pares distintos",
        "`load_annotations`; el .txt sin .wav se descarta",
    ),
    (
        "Normalización de etiquetas",
        f"{n_review:,} fuera del\nvocabulario",
        "sinónimos y `MANUAL_FIXES`",
    ),
    (
        "Saneamiento geométrico",
        f"recorte a Nyquist\n({P.f_max / 1000:.1f} kHz)",
        "fuera las cajas degeneradas",
    ),
    (
        "Vocabulario cerrado",
        f"{n_no_phrase:,} sin las 3\nclases de frase",
        "y los trinos de lw fusionados",
    ),
    (
        "Selección del subconjunto",
        f"{n_experiment:,} anotaciones\n{len(CLASSES)} clases",
        f"pares con >= {MIN_PAIR_COUNT}",
    ),
]
WINDOW_STEPS = [
    (
        "Ventaneo",
        f"{len(manifest):,} ventanas" if manifest else "ventanas de 3 s",
        f"clip {P.clip_len_s:g} s, salto {P.clip_hop_s:g} s",
    ),
    ("Partición por grabación", "60 / 22,5 / 17,5 %", "ninguna ventana cruza"),
    ("Caché de tensores", "train / val / test .pt", "mel de potencia y cajas cxcywh"),
]

fig, ax = canvas(12.6, 8.4)
TOP = 100 * 8.4 / 12.6
ROWS = [STEPS[:3], STEPS[3:], WINDOW_STEPS]
row_y = [TOP - 13, TOP - 36, TOP - 59]
placed = []
for r, (row, cy) in enumerate(zip(ROWS, row_y)):
    edge = SERIES[2] if r == 2 else BLUE
    placed.append(
        flow_row(
            ax,
            [{"title": name, "body": count, "foot": note} for name, count, note in row],
            cy,
            margin=6.5,
            gap=3.4,
            wrap=24,
            fill=PANEL if r == 2 else SURFACE,
            edge=edge,
            arrow_color=edge,
            lw=1.3,
            fs=9.0,
            body_fs=7.8,
            foot_fs=6.9,
        )
    )
for r in range(2):
    cx_last, cy_last, w_last, h_last = placed[r][-1]
    cx_first, cy_next, w_first, h_next = placed[r + 1][0]
    mid = (cy_last - h_last / 2 + cy_next + h_next / 2) / 2
    elbow(
        ax,
        [
            (cx_last, cy_last - h_last / 2),
            (cx_last, mid),
            (cx_first, mid),
            (cx_first, cy_next + h_next / 2),
        ],
        color=SERIES[2] if r == 1 else BLUE,
    )

chip(ax, 97.4, row_y[0], "RE1.1", BLUE, fs=7.5)
chip(ax, 97.4, row_y[2], "RE1.2", SERIES[2], fs=7.5)
save(fig, "curation_pipeline")
plt.show()

In [ ]:
fig, ax = canvas(12.8, 6.8)
TOP = 100 * 6.8 / 12.8
SPAN_S = 12.0
X0, TIMELINE_W = 3.0, 62.0
SCALE = TIMELINE_W / SPAN_S
NOTE_X = X0 + TIMELINE_W + 3.0
AXIS_Y = TOP - 16


def t2x(t):
    return X0 + t * SCALE


starts = np.arange(0, SPAN_S - P.clip_len_s + 0.01, P.clip_hop_s)
for i, start in enumerate(starts):
    y = AXIS_Y + 3.0 + (i % 2) * 4.0
    ax.add_patch(
        FancyBboxPatch(
            (t2x(start), y),
            P.clip_len_s * SCALE,
            3.2,
            boxstyle="round,pad=0,rounding_size=0.7",
            facecolor=BLUE,
            alpha=0.16 + 0.06 * (i % 2),
            edgecolor=BLUE,
            linewidth=1.0,
        )
    )
    caption_at(ax, t2x(start + P.clip_len_s / 2), y + 1.6, f"w{i}", fs=6.8, color=BLUE)
caption_at(
    ax,
    NOTE_X,
    AXIS_Y + 5.0,
    f"ventanas de {P.clip_len_s:g} s\ncon salto de {P.clip_hop_s:g} s",
    fs=7.8,
    color=BLUE,
    ha="left",
    weight="bold",
)

ax.plot([X0, X0 + SPAN_S * SCALE], [AXIS_Y, AXIS_Y], color=BASELINE, linewidth=1.2)
for t in range(0, int(SPAN_S) + 1):
    ax.plot([t2x(t), t2x(t)], [AXIS_Y - 0.8, AXIS_Y], color=BASELINE, linewidth=0.9)
    if t % 3 == 0:
        caption_at(ax, t2x(t), AXIS_Y - 3.0, f"{t} s", fs=7.5)

EVENTS = [
    (0.6, 1.5, "Dentro de una ventana", GOOD, "entra completo en w0"),
    (
        4.2,
        5.6,
        "A caballo de dos",
        GOOD,
        "entra en w2 y en w3: la misma anotación produce dos cajas",
    ),
    (
        7.3,
        8.0,
        "Cerca del borde",
        WARNING,
        f"sólo entra donde el solape alcanza min_overlap = {P.min_overlap:g}",
    ),
    (
        2.0,
        10.5,
        "Más largo que dos clips",
        CRITICAL,
        f"nunca alcanza {P.min_overlap:g} de su duración en ninguna ventana: inalcanzable",
    ),
]
pitch = (AXIS_Y - 7.0) / len(EVENTS)
for i, (t0, t1, name, color, note) in enumerate(EVENTS):
    y = AXIS_Y - 7.0 - i * pitch
    ax.add_patch(
        FancyBboxPatch(
            (t2x(t0), y),
            (t1 - t0) * SCALE,
            3.2,
            boxstyle="round,pad=0,rounding_size=0.7",
            facecolor=color,
            alpha=0.22,
            edgecolor=color,
            linewidth=1.4,
        )
    )
    caption_at(ax, t2x(t0) + 0.8, y + 1.6, f"{t1 - t0:.1f} s", fs=7, color=INK_2, ha="left")
    caption_at(
        ax, NOTE_X, y + 3.4, f"{name}\n{note}", fs=7.4, color=color, ha="left", wrap=30, va="top"
    )

save(fig, "windowing")
plt.show()

In [ ]:
fig, ax = canvas(12.0, 4.8)
TOP = 100 * 4.8 / 12.0
BLOCK_Y = TOP - 19

for x0, title, ok in [(3, "Partición por ventana", False), (54, "Partición por grabación", True)]:
    caption_at(ax, x0, TOP - 4, title, fs=10, color=INK, ha="left", weight="bold")
    width = 40
    ax.add_patch(
        FancyBboxPatch(
            (x0, BLOCK_Y),
            width,
            6,
            boxstyle="round,pad=0,rounding_size=0.8",
            facecolor=PANEL,
            edgecolor=BASELINE,
            linewidth=1.1,
        )
    )
    caption_at(ax, x0 + width - 4, BLOCK_Y - 3.0, "una grabación", fs=7.5, ha="right")
    assignment = ["train"] * 4 if ok else ["train", "test", "train", "val"]
    tint = {"train": BLUE, "val": SERIES[1], "test": SERIES[2]}
    for i in range(4):
        start = x0 + 1.5 + i * 8.5
        ax.add_patch(
            FancyBboxPatch(
                (start, BLOCK_Y + 1 + (i % 2) * 2),
                14,
                2.6,
                boxstyle="round,pad=0,rounding_size=0.5",
                facecolor=tint[assignment[i]],
                alpha=0.6,
                edgecolor=tint[assignment[i]],
                linewidth=1.0,
            )
        )
        caption_at(
            ax,
            start + 7,
            BLOCK_Y + 2.3 + (i % 2) * 2,
            assignment[i],
            fs=6.4,
            color=SURFACE if assignment[i] == "train" else INK,
        )
    if not ok:
        ax.add_patch(
            FancyBboxPatch(
                (x0 + 9.5, BLOCK_Y - 1.5),
                6,
                9,
                boxstyle="round,pad=0,rounding_size=0.5",
                facecolor="none",
                edgecolor=CRITICAL,
                linewidth=1.4,
                linestyle=(0, (3, 2)),
            )
        )
        caption_at(
            ax,
            x0 + 12.5,
            BLOCK_Y - 6.5,
            "audio compartido\nentre particiones",
            fs=7,
            color=CRITICAL,
            va="top",
        )

save(fig, "split_by_recording")
plt.show()

In [ ]:
NESTED_PAIRS = [("lw/cc", "lw/cs"), ("sm/fc", "sm/fs"), ("sb/pcs", "sb/ppc")]

nesting_rows = []
for phrase_pair, syllable_pair in NESTED_PAIRS:
    phrases = annotations[annotations.pair == phrase_pair]
    syllables = annotations[annotations.pair == syllable_pair]
    if phrases.empty or syllables.empty:
        continue
    inside = 0
    for audio_path, group in syllables.groupby("audio_path"):
        window = phrases[phrases.audio_path == audio_path]
        if window.empty:
            continue
        mid = ((group.begin_time_s + group.end_time_s) / 2).to_numpy()[:, None]
        starts = window.begin_time_s.to_numpy()[None, :]
        ends = window.end_time_s.to_numpy()[None, :]
        inside += int(((mid >= starts) & (mid <= ends)).any(axis=1).sum())
    with_syllable = 0
    for audio_path, group in phrases.groupby("audio_path"):
        window = syllables[syllables.audio_path == audio_path]
        if window.empty:
            continue
        mid = ((window.begin_time_s + window.end_time_s) / 2).to_numpy()[None, :]
        starts = group.begin_time_s.to_numpy()[:, None]
        ends = group.end_time_s.to_numpy()[:, None]
        with_syllable += int(((mid >= starts) & (mid <= ends)).any(axis=1).sum())
    nesting_rows.append(
        {
            "frase": phrase_pair,
            "sílaba": syllable_pair,
            "frases": len(phrases),
            "frases con >=1 sílaba dentro": with_syllable / len(phrases),
            "sílabas": len(syllables),
            "sílabas dentro de una frase": inside / len(syllables),
        }
    )
nesting = pd.DataFrame(nesting_rows)

phrase_pair, syllable_pair = NESTED_PAIRS[0]
phrases = annotations[annotations.pair == phrase_pair]
best = None
for r in phrases.itertuples():
    contained = annotations[
        (annotations.audio_path == r.audio_path)
        & (annotations.pair == syllable_pair)
        & (annotations.begin_time_s >= r.begin_time_s - 0.05)
        & (annotations.end_time_s <= r.end_time_s + 0.05)
    ]
    if best is None or len(contained) > len(best[1]):
        best = (r, contained)
phrase, syllables = best

pad = max(phrase.duration_s * 0.18, 0.25)
start_s = max(phrase.begin_time_s - pad, 0.0)
duration_s = phrase.duration_s + 2 * pad
waveform = load_segment(phrase.audio_path, start_s, duration_s)

fig, ax = plt.subplots(figsize=(10.4, 4.8))
draw_spectrogram(ax, log_mel(waveform), duration_s)
draw_box(
    ax,
    phrase.begin_time_s - start_s,
    phrase.end_time_s - start_s,
    phrase.low_freq_hz,
    phrase.high_freq_hz,
    color=SERIES[1],
    label=f"{phrase_pair.upper()}  (frase, {phrase.duration_s:.1f} s)",
    lw=2.0,
    dashed=True,
)
for r in syllables.itertuples():
    draw_box(
        ax,
        r.begin_time_s - start_s,
        r.end_time_s - start_s,
        r.low_freq_hz,
        r.high_freq_hz,
        color=GOOD,
        lw=1.4,
    )
ax.text(
    (phrase.begin_time_s - start_s + phrase.end_time_s - start_s) / 2,
    0.06,
    f"{len(syllables)} sílabas {syllable_pair.upper()} dentro de una sola frase",
    ha="center",
    va="center",
    fontsize=8.5,
    color=GOOD,
    fontweight="bold",
    zorder=6,
    bbox=dict(boxstyle="round,pad=0.35", facecolor=SURFACE, edgecolor="none", alpha=0.92),
)
phrase_share = nesting.set_index("frase").loc[phrase_pair, "frases con >=1 sílaba dentro"]
save(fig, "nesting_phrase")
plt.show()

In [ ]:
N_COLS = 5
n_rows = int(np.ceil(len(CLASSES) / N_COLS))
order = experiment.groupby("pair").low_freq_hz.median().sort_values().index.tolist()

fig, axes = plt.subplots(n_rows, N_COLS, figsize=(3.0 * N_COLS, 2.35 * n_rows))
axes_flat = axes.ravel()
for ax, pair in zip(axes_flat, order):
    row = pick_annotation(pair, df=experiment)
    span = max(row.duration_s * 2.4, 0.35)
    start_s = max(row.begin_time_s - (span - row.duration_s) / 2, 0.0)
    try:
        waveform = load_segment(row.audio_path, start_s, span)
        draw_spectrogram(ax, log_mel(waveform), span, n_ticks=4)
        draw_box(
            ax,
            row.begin_time_s - start_s,
            row.end_time_s - start_s,
            row.low_freq_hz,
            row.high_freq_hz,
            color=GOOD,
            lw=1.3,
        )
    except Exception as exc:  # noqa: BLE001
        ax.text(0.5, 0.5, f"sin audio\n{exc}", ha="center", va="center", fontsize=7, color=CRITICAL)
    ax.set_title(f"{pair.upper()}  ·  {CALL_NAME.get(pair, '')}", fontsize=8.5, pad=14, loc="left")
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.tick_params(labelsize=6.5)
    ax.text(
        0,
        1.015,
        f"n={int((experiment.pair == pair).sum()):,} anotaciones · mediana {row.duration_s * 1000:.0f} ms",
        transform=ax.transAxes,
        fontsize=6.8,
        color=INK_2,
        va="bottom",
    )

for ax in axes_flat[len(order) :]:
    ax.axis("off")

fig.tight_layout(rect=(0, 0, 1, 0.945))
save(fig, "call_gallery")
plt.show()

In [ ]:
DETR_ARCHITECTURE = "ast_deformable_detr"


def pick_checkpoint():
    """El .pth del DETR cuyo número de clases coincide con el `labels.json` del caché.

    Filtra por arquitectura: en `checkpoints/` ahora conviven los tres detectores, y ni
    Faster R-CNN ni YOLO tienen `dim` ni `n_queries` que dibujar. Los checkpoints
    anteriores al registro no traen la clave `architecture`, y todos eran del DETR.
    """
    candidates = sorted(CHECKPOINT_DIR.glob("*.pth"))
    if not candidates:
        return None
    wanted = len(CLASSES)
    scored = []
    for path in candidates:
        try:
            head = torch.load(path, map_location="cpu", weights_only=False)
        except Exception:
            continue
        if head.get("architecture", DETR_ARCHITECTURE) != DETR_ARCHITECTURE:
            continue
        scored.append((len(head.get("labels", [])) == wanted, path.stat().st_mtime, path))
    scored.sort(reverse=True)
    return scored[0][2] if scored else None


CHECKPOINT = pick_checkpoint()
if CHECKPOINT is not None:
    meta = torch.load(CHECKPOINT, map_location="cpu", weights_only=False)
    # El registro agrupa la geometría en `hparams`; antes iba suelta en la raíz. Sin
    # este fallback, cualquier checkpoint entrenado desde el refactor rompe la figura.
    hparams = meta.get("hparams") or meta
    ARCH = {
        "dim": hparams["dim"],
        "n_queries": hparams["n_queries"],
        "n_levels": hparams.get("n_levels", 3),
        "n_frames": hparams.get("n_frames", P.n_frames),
        "time_stride": hparams.get("time_stride", 5),
        "frontend": hparams.get("frontend", "pcen"),
        # `n_decoder_layers` no viaja en el checkpoint: es el default de
        # `DeformableDETR` y se declara acá para que la figura no lo invente.
        "layers": 6,
        "n_classes": len(meta["labels"]),
        "labels": list(meta["labels"]),
        "epoch": meta.get("epoch"),
        "recall_agn": meta.get("recall_agn"),
        "precision_agn": meta.get("precision_agn"),
        "config": meta.get("config", {}),
    }
    del meta
else:
    ARCH = {
        "dim": 128,
        "n_queries": 64,
        "n_levels": 3,
        "n_frames": P.n_frames,
        "time_stride": 2,
        "frontend": "pcen",
        "layers": 6,
        "n_classes": len(CLASSES),
        "labels": CLASSES,
        "epoch": None,
        "recall_agn": None,
        "precision_agn": None,
        "config": {},
    }

PATCH, FREQ_STRIDE = 16, 10
FREQ_OUT = (P.n_mels - PATCH) // FREQ_STRIDE + 1
TIME_OUT = (ARCH["n_frames"] - PATCH) // ARCH["time_stride"] + 1
LEVELS = (
    [(FREQ_OUT * 2, TIME_OUT * 2), (FREQ_OUT, TIME_OUT), (FREQ_OUT // 2, TIME_OUT // 2)][
        3 - ARCH["n_levels"] :
    ]
    if ARCH["n_levels"] <= 3
    else []
)
LEVELS = [(FREQ_OUT * 2, TIME_OUT * 2), (FREQ_OUT, TIME_OUT), (FREQ_OUT // 2, TIME_OUT // 2)][
    : ARCH["n_levels"]
]

In [ ]:
# Dos pisos, como en la literatura de detección: arriba el recorrido del dato de punta a
# punta, abajo los dos bloques que no se entienden como una caja --el parcheo del AST y
# la capa del decodificador-- abiertos por dentro. Todas las formas salen de `P` y de
# `ARCH`, así que la figura sigue al checkpoint que se haya entrenado.

fig, ax = canvas(13.4, 8.6)
TOP = 100 * 8.6 / 13.4

FLOW_Y = TOP - 12.5  # eje del piso de arriba
NAME_Y = FLOW_Y + 8.6  # nombre de la etapa
CHIP_Y = NAME_Y + 4.6  # entrenable / congelado, todos a la misma altura
SHAPE_Y = FLOW_Y - 7.8  # forma del tensor, va="top"
PANEL_CY, PANEL_H = 15.8, 28.0

mel_shape = f"(1, {P.n_mels}, {P.n_frames})"
token_shape = f"({FREQ_OUT}x{TIME_OUT}, 768)"

# --- Piso 1: el recorrido del dato --------------------------------------------
slab(ax, 7.5, FLOW_Y, 11.0, 10.0)
caption_at(ax, 7.5, NAME_Y, "Espectrograma\nmel", fs=8.0, color=INK, weight="bold")
caption_at(ax, 7.5, SHAPE_Y, mel_shape, fs=6.6, color=INK_2, va="top")

node(ax, 23.0, FLOW_Y, 12.0, 9.0, "PCEN\nentrenable", fill=SURFACE, edge=BLUE, lw=1.3, fs=8.2)
chip(ax, 23.0, CHIP_Y, "entrenable", GOOD, fs=6.4)
caption_at(ax, 23.0, SHAPE_Y, mel_shape, fs=6.6, color=INK_2, va="top")

node(ax, 40.0, FLOW_Y, 13.0, 9.0, "AST\ncongelado", fill=PANEL, edge=MUTED, lw=1.3, fs=8.2)
chip(ax, 40.0, CHIP_Y, "congelado", MUTED, fs=6.4)
caption_at(ax, 40.0, SHAPE_Y, "ViT-base, AudioSet", fs=6.6, color=INK_2, va="top")

token_strip(ax, 52.0, FLOW_Y, 2.6, 10.0, 7)
caption_at(ax, 52.0, NAME_Y, "Tokens\nlatentes", fs=8.0, color=INK, weight="bold")
caption_at(ax, 52.0, SHAPE_Y, token_shape, fs=6.6, color=INK_2, va="top")

feature_pyramid(ax, 63.0, FLOW_Y, 12.0, 11.0, LEVELS, label_y=SHAPE_Y)
caption_at(ax, 63.0, NAME_Y, "Pirámide\nmultiescala", fs=8.0, color=INK, weight="bold")
chip(ax, 63.0, CHIP_Y, "entrenable", GOOD, fs=6.4)

node(
    ax,
    79.0,
    FLOW_Y,
    14.0,
    9.0,
    "Decodificador\ndeformable",
    fill=SURFACE,
    edge=BLUE,
    lw=1.3,
    fs=8.0,
)
chip(ax, 79.0, CHIP_Y, "entrenable", GOOD, fs=6.4)

detection_slab(
    ax,
    94.0,
    FLOW_Y,
    10.0,
    10.0,
    [(0.32, 0.68, 0.34, 0.20), (0.72, 0.72, 0.24, 0.16), (0.46, 0.30, 0.46, 0.24)],
)
caption_at(ax, 94.0, NAME_Y, "Cajas\netiquetadas", fs=8.0, color=INK, weight="bold")
caption_at(ax, 94.0, SHAPE_Y, "tiempo x frecuencia\n+ clase", fs=6.6, color=INK_2, va="top")

for x0, x1 in ((13.0, 16.8), (29.2, 33.3), (46.6, 50.5), (53.4, 56.8), (69.2, 71.8), (86.2, 88.8)):
    arrow(ax, (x0, FLOW_Y), (x1, FLOW_Y), color=INK, lw=1.3)

# --- Puentes al segundo piso ---------------------------------------------------
zoom_arrow(ax, 40.0, FLOW_Y - 13.2, 6.6, 6.6)
zoom_arrow(ax, 79.0, FLOW_Y - 13.2, 6.6, 6.6)

# --- Piso 2a: cómo el AST parte el espectrograma -------------------------------
detail_panel(ax, 25.5, PANEL_CY, 47.0, PANEL_H, "AST: parcheo y codificación")
inner = PANEL_CY - 2.4

slab(ax, 8.0, inner, 7.0, 9.0)
caption_at(ax, 8.0, inner - 6.6, "mel", fs=6.4, color=INK_2, va="top")

patch_grid(ax, 19.0, inner, 9.0, 9.0, 4, 3)
caption_at(
    ax,
    19.0,
    inner - 6.6,
    f"parches {PATCH}x{PATCH}\npaso ({FREQ_STRIDE}, {ARCH['time_stride']})",
    fs=6.4,
    color=INK_2,
    va="top",
)

token_strip(ax, 28.5, inner, 2.4, 10.0, 6)
node(ax, 35.5, inner, 5.0, 11.0, "Encoder\nViT", fill=PANEL, edge=MUTED, lw=1.1, fs=6.8, wrap=8)
token_strip(ax, 43.0, inner, 2.4, 10.0, 6)
caption_at(ax, 43.0, inner - 6.6, token_shape, fs=6.4, color=INK_2, va="top")

for x0, x1 in ((11.8, 14.2), (23.8, 27.1), (29.9, 32.8), (38.2, 41.6)):
    arrow(ax, (x0, inner), (x1, inner), color=INK, lw=1.1)

# --- Piso 2b: qué hace el decodificador con las queries ------------------------
detail_panel(ax, 74.5, PANEL_CY, 47.0, PANEL_H, "Capa del decodificador deformable")

QUERY_Y = PANEL_CY + 5.6
LAYER_Y = PANEL_CY - 1.2
FEATURE_Y = PANEL_CY - 8.4
JOIN_X = 66.5

token_strip(ax, 59.0, QUERY_Y, 11.0, 2.4, 6, vertical=False)
caption_at(ax, 59.0, QUERY_Y + 3.2, f"{ARCH['n_queries']} object queries", fs=6.4, color=INK_2)

feature_pyramid(ax, 59.0, FEATURE_Y, 10.0, 5.2, LEVELS, label_y=FEATURE_Y - 3.4)
caption_at(ax, 59.0, FEATURE_Y + 4.2, "features multiescala", fs=6.4, color=INK_2)

# Las dos entradas se juntan en una sola flecha: dos puntas encimadas sobre el mismo
# borde se leen como un error de dibujo, no como dos caminos.
for y0 in (QUERY_Y, FEATURE_Y):
    ax.plot([64.8, JOIN_X, JOIN_X], [y0, y0, LAYER_Y], color=INK, lw=1.1, zorder=1)
arrow(ax, (JOIN_X, LAYER_Y), (68.4, LAYER_Y), color=INK, lw=1.1)

node(
    ax,
    76.0,
    LAYER_Y,
    15.0,
    11.5,
    f"Capa x{ARCH['layers']}",
    "auto-atención\natención deformable\nFFN",
    fill=SURFACE,
    edge=BLUE,
    lw=1.2,
    fs=7.4,
    body_fs=6.4,
    wrap=20,
)
node(ax, 87.5, LAYER_Y, 4.2, 10.0, "Cabeza", fill=PANEL, edge=MUTED, lw=1.1, fs=6.6, wrap=7)
token_strip(ax, 93.5, LAYER_Y, 2.3, 8.5, 5)
caption_at(
    ax,
    93.5,
    LAYER_Y - 5.8,
    f"clase ({ARCH['n_classes']}+1)\ny caja",
    fs=6.4,
    color=INK_2,
    va="top",
)

arrow(ax, (83.6, LAYER_Y), (85.3, LAYER_Y), color=INK, lw=1.1)
arrow(ax, (89.7, LAYER_Y), (92.2, LAYER_Y), color=INK, lw=1.1)

# El refinamiento iterativo: la caja que emite una capa es la referencia de la siguiente.
ax.annotate(
    "",
    xy=(76.0, LAYER_Y + 6.0),
    xytext=(87.5, LAYER_Y + 5.2),
    zorder=2,
    arrowprops=dict(
        arrowstyle="-|>",
        color=MUTED,
        lw=1.0,
        shrinkA=2,
        shrinkB=2,
        connectionstyle="arc3,rad=-0.35",
        mutation_scale=10,
    ),
)
caption_at(ax, 81.5, LAYER_Y + 9.0, "refina la caja de referencia", fs=6.2, color=MUTED)

save(fig, "architecture")
plt.show()

In [ ]:
cell_ms = [P.clip_len_s * 1000 / t for _, t in LEVELS]
durations = experiment.groupby("pair").duration_s.median().sort_values() * 1000

fig, axes = plt.subplots(
    1, 2, figsize=(12.6, 5.6), gridspec_kw={"width_ratios": [0.85, 1.25], "wspace": 0.30}
)

ax = axes[0]
ax.set_xlim(0, 100)
ax.set_ylim(0, 100)
ax.axis("off")
for i, ((f, t), ms) in enumerate(zip(LEVELS, cell_ms)):
    y = 70 - i * 25
    ax.add_patch(
        FancyBboxPatch(
            (4, y - 9),
            56,
            17,
            boxstyle="round,pad=0,rounding_size=1.2",
            facecolor=SURFACE,
            edgecolor=BLUE,
            linewidth=1.3,
        )
    )
    cols = max(4, int(round(40 * t / LEVELS[0][1])))
    rows = max(2, int(round(12 * f / LEVELS[0][0])))
    for c in range(1, cols):
        ax.plot([4 + 56 * c / cols] * 2, [y - 9, y + 8], color=BLUE, linewidth=0.4, alpha=0.45)
    for r in range(1, rows):
        ax.plot([4, 60], [y - 9 + 17 * r / rows] * 2, color=BLUE, linewidth=0.4, alpha=0.45)
    ax.text(
        64,
        y + 2.5,
        f"nivel {i}: {f} x {t}",
        fontsize=9.5,
        fontweight="bold",
        color=INK,
        va="center",
    )
    ax.text(64, y - 3.5, f"una celda = {ms:.0f} ms", fontsize=8.2, color=INK_2, va="center")

ax = axes[1]
ys = np.arange(len(durations))
ax.set_xscale("log")
ax.set_xlim(5, max(durations.max() * 1.9, 4000))
ax.set_ylim(len(durations) - 0.4, -2.6)
for y, (pair, value) in zip(ys, durations.items()):
    ax.plot([5, value], [y, y], color=GRID, linewidth=1.0, zorder=1)
    ax.scatter([value], [y], s=26, color=BLUE, zorder=3)
    ax.text(value * 1.10, y, f"{value:.0f} ms", va="center", fontsize=7, color=INK_2)
for i, ms in enumerate(cell_ms):
    ax.axvline(ms, color=SERIES[1], linewidth=1.2, linestyle=(0, (4, 3)), zorder=2)
    ax.text(
        ms,
        -0.5 - i * 0.62,
        f"nivel {i}: {ms:.0f} ms",
        color=SERIES[1],
        fontsize=7.5,
        ha="left",
        va="center",
    )
ax.set_yticks(ys, durations.index, fontsize=8)
ax.set_xlabel("duración mediana de la clase (ms, escala log)")
style_value_axis(ax, "x")

save(
    fig,
    "multiscale_pyramid",
    data={
        "niveles": [{"f": f, "t": t, "celda_ms": ms} for (f, t), ms in zip(LEVELS, cell_ms)],
        "clase": durations.index,
        "duracion_mediana_ms": durations,
    },
)
plt.show()

In [ ]:
from models.deformable_detr import DeformableAttention  # noqa: E402

attention = DeformableAttention(dim=ARCH["dim"], n_heads=8, n_points=4, n_levels=ARCH["n_levels"])
offsets = attention.offsets.bias.detach().view(8, ARCH["n_levels"], 4, 2).numpy()

REFERENCES = [
    ((0.30, 0.62, 0.06, 0.10), "llamada corta (50 ms)"),
    ((0.62, 0.40, 0.55, 0.34), "llamada larga (1,6 s)"),
]

fig, axes = plt.subplots(1, 2, figsize=(11.4, 5.0), gridspec_kw={"wspace": 0.16})
for ax, (ref, name) in zip(axes, REFERENCES):
    cx, cy, w, h = ref
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_aspect("equal")
    ax.add_patch(Rectangle((0, 0), 1, 1, facecolor=PANEL, edgecolor=BASELINE, linewidth=1.0))
    ax.add_patch(
        Rectangle(
            (cx - w / 2, cy - h / 2),
            w,
            h,
            facecolor=BLUE,
            alpha=0.13,
            edgecolor=BLUE,
            linewidth=1.6,
        )
    )
    ax.scatter([cx], [cy], s=30, color=BLUE, zorder=5)
    points = offsets[:, 0] / 4.0 * np.array([w, h]) * 0.5 + np.array([cx, cy])
    ax.scatter(
        points[..., 0].ravel(),
        points[..., 1].ravel(),
        s=20,
        marker="o",
        facecolor="none",
        edgecolors=SERIES[1],
        linewidths=1.1,
        zorder=4,
    )
    ax.set_xticks([])
    ax.set_yticks([])
    for side in ("top", "right", "left", "bottom"):
        ax.spines[side].set_visible(False)
    ax.set_title(name, fontsize=10, pad=8)

save(fig, "deformable_sampling")
plt.show()

In [ ]:
fig, ax = canvas(12.4, 5.6)
TOP = 100 * 5.6 / 12.4
cy = TOP * 0.60
blocks = flow_row(
    ax,
    [
        {
            "title": "Query",
            "body": f"{ARCH['n_queries']} vectores\naprendidos",
            "foot": f"({ARCH['n_queries']}, {ARCH['dim']})",
        },
        {
            "title": "Auto-atención",
            "body": "las queries se reparten\nel trabajo entre sí",
            "foot": "+ query_pos en cada capa",
        },
        {
            "title": "Atención deformable",
            "body": "muestrea los 3 niveles\nalrededor de la caja",
            "foot": "8 cabezas x 4 puntos",
        },
        {
            "title": "FFN",
            "body": f"{ARCH['dim']} -> 1024 -> {ARCH['dim']}\ncon LayerNorm residual",
            "foot": "una por capa",
        },
        {
            "title": "Cabezas",
            "body": "clase y delta de caja\nen espacio logit",
            "foot": f"({ARCH['n_classes']}+1) y 4",
        },
    ],
    cy,
    margin=2.5,
    gap=2.6,
    wrap=20,
    fill=SURFACE,
    edge=BLUE,
    lw=1.3,
    fs=8.6,
    body_fs=7.2,
    foot_fs=6.8,
)
h = blocks[0][3]
caption_at(ax, blocks[0][0], cy + h / 2 + 3.2, "x 6 capas", fs=8, color=INK_2)
elbow(
    ax,
    [
        (blocks[-1][0], cy - h / 2),
        (blocks[-1][0], cy - h / 2 - 6),
        (blocks[2][0], cy - h / 2 - 6),
        (blocks[2][0], cy - h / 2 - 1),
    ],
    color=SERIES[1],
)
caption_at(
    ax,
    blocks[2][0],
    cy - h / 2 - 8.5,
    "refinamiento iterativo",
    fs=7.6,
    color=SERIES[1],
    wrap=90,
    va="top",
)
save(fig, "decoder_layer")
plt.show()

In [ ]:
from scipy.optimize import linear_sum_assignment  # noqa: E402

from models.criterion import HungarianMatcher  # noqa: E402

truth_boxes = torch.tensor(
    [[0.20, 0.62, 0.10, 0.14], [0.52, 0.58, 0.14, 0.18], [0.80, 0.30, 0.20, 0.22]],
    dtype=torch.float32,
)
truth_labels = torch.tensor([0, 0, 1])
predicted_boxes = torch.tensor(
    [
        [0.21, 0.63, 0.12, 0.15],  # buena sobre la primera
        [0.50, 0.57, 0.20, 0.26],  # laxa sobre la segunda
        [0.79, 0.31, 0.19, 0.20],  # buena sobre la tercera
        [0.35, 0.20, 0.10, 0.10],  # fondo
        [0.65, 0.80, 0.16, 0.12],  # fondo
    ],
    dtype=torch.float32,
)
logits = torch.full((5, 3), -2.0)
logits[0, 0] = logits[2, 1] = 3.0
logits[1, 0] = 1.2
logits[3, 2] = logits[4, 2] = 2.5  # el canal de no-objeto

matcher = HungarianMatcher()
outputs = {"pred_logits": logits[None], "pred_boxes": predicted_boxes[None]}
targets = [{"labels": truth_labels, "boxes": truth_boxes}]
query_index, target_index = matcher(outputs, targets)[0]

probabilities = logits.softmax(-1)
cost_class = -probabilities[:, truth_labels]
cost_bbox = torch.cdist(predicted_boxes, truth_boxes, p=1)
from torchvision.ops import box_convert, generalized_box_iou  # noqa: E402

cost_iou = -generalized_box_iou(
    box_convert(predicted_boxes, "cxcywh", "xyxy"), box_convert(truth_boxes, "cxcywh", "xyxy")
)
cost = (
    matcher.cost_class * cost_class + matcher.cost_bbox * cost_bbox + matcher.cost_iou * cost_iou
).numpy()

fig, axes = plt.subplots(
    1, 2, figsize=(12.0, 5.0), gridspec_kw={"width_ratios": [1, 1.15], "wspace": 0.24}
)

ax = axes[0]
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_aspect("equal")
ax.add_patch(Rectangle((0, 0), 1, 1, facecolor=PANEL, edgecolor=BASELINE, linewidth=1.0))
for i, box in enumerate(truth_boxes.numpy()):
    cx, cy, w, h = box
    ax.add_patch(
        Rectangle((cx - w / 2, cy - h / 2), w, h, facecolor="none", edgecolor=GOOD, linewidth=2.0)
    )
    ax.text(cx - w / 2, cy + h / 2 + 0.02, f"g{i}", color=GOOD, fontsize=8.5, fontweight="bold")
for j, box in enumerate(predicted_boxes.numpy()):
    cx, cy, w, h = box
    matched = j in query_index.tolist()
    ax.add_patch(
        Rectangle(
            (cx - w / 2, cy - h / 2),
            w,
            h,
            facecolor="none",
            edgecolor=BLUE if matched else MUTED,
            linewidth=1.6 if matched else 1.1,
            linestyle="-" if matched else (0, (3, 2)),
        )
    )
    ax.text(
        cx + w / 2,
        cy - h / 2 - 0.03,
        f"q{j}",
        color=BLUE if matched else MUTED,
        fontsize=8.5,
        ha="right",
    )
ax.set_xticks([])
ax.set_yticks([])
for side in ("top", "right", "left", "bottom"):
    ax.spines[side].set_visible(False)

ax = axes[1]
image = ax.imshow(cost, cmap="Blues_r", aspect="auto")
for j in range(cost.shape[0]):
    for i in range(cost.shape[1]):
        chosen = (j, i) in list(zip(query_index.tolist(), target_index.tolist()))
        ax.text(
            i,
            j,
            f"{cost[j, i]:.2f}",
            ha="center",
            va="center",
            fontsize=8.5,
            color=INK if cost[j, i] > cost.min() + (cost.max() - cost.min()) * 0.45 else SURFACE,
            fontweight="bold" if chosen else "normal",
        )
        if chosen:
            ax.add_patch(
                Rectangle(
                    (i - 0.5, j - 0.5), 1, 1, facecolor="none", edgecolor=SERIES[1], linewidth=2.4
                )
            )
ax.set_xticks(
    range(3),
    [f"g{i}\n{CLASSES[label]}" for i, label in enumerate(truth_labels.tolist())],
    fontsize=8,
)
ax.set_yticks(range(5), [f"q{j}" for j in range(5)], fontsize=8)
for side in ("top", "right", "left", "bottom"):
    ax.spines[side].set_visible(False)

save(fig, "hungarian_matching")
plt.show()

In [ ]:
from train import TRAINERS  # noqa: E402

CONFIG = TRAINERS["detr"][1]

fig, ax = canvas(13.2, 4.4)
TOP = 100 * 4.4 / 13.2
cy = TOP * 0.60
boxes = flow_row(
    ax,
    [
        {
            "title": "Caché de tensores",
            "body": "train.pt / val.pt\nmel de potencia + cajas",
            "foot": "`SpectrogramDataset`",
        },
        {
            "title": "Aumento",
            "body": f"jitter de cajas ±{CONFIG.jitter.time_scale:.0%}\ny corrimiento ±{CONFIG.jitter.time_shift:.0%}",
            "foot": "sólo en entrenamiento",
        },
        {
            "title": "Modelo",
            "body": f"lotes de {CONFIG.batch_size}\nAdamW lr {CONFIG.learning_rate:g}, wd {CONFIG.weight_decay:g}",
            "foot": f"OneCycle, {CONFIG.epochs} épocas",
        },
        {
            "title": "SetCriterion",
            "body": "húngaro + clase, L1 y GIoU\nen las 6 capas",
            "foot": "eos_coef 0,1 para el no-objeto",
        },
        {
            "title": "Validación por época",
            "body": f"recall e IoU a IoU {CONFIG.iou_threshold:g}\ncon score ≥ {CONFIG.score_threshold:g}",
            "foot": f"NMS IoU {CONFIG.nms_iou:g}",
        },
        {
            "title": "Selección",
            "body": f"F-β con β = {CONFIG.beta:g}\n(el recall pesa 9x)",
            "foot": "guarda el mejor .pth",
        },
    ],
    cy,
    margin=2.0,
    gap=2.0,
    wrap=20,
    fill=SURFACE,
    edge=BLUE,
    lw=1.3,
    fs=8.4,
    body_fs=7.0,
    foot_fs=6.7,
)
h = boxes[0][3]
elbow(
    ax,
    [
        (boxes[-1][0], cy - h / 2),
        (boxes[-1][0], cy - h / 2 - 5),
        (boxes[2][0], cy - h / 2 - 5),
        (boxes[2][0], cy - h / 2 - 1),
    ],
    color=SERIES[1],
)
caption_at(ax, boxes[2][0], cy - h / 2 - 7.5, "una época", fs=7.4, color=SERIES[1], va="top")
save(fig, "training_pipeline")
plt.show()

In [ ]:
fig, ax = canvas(13.2, 4.4)
TOP = 100 * 4.4 / 13.2
cy = TOP * 0.60
boxes = flow_row(
    ax,
    [
        {"title": "Grabación .wav", "body": "de cualquier duración", "foot": "`wav_paths`"},
        {
            "title": "Ventaneo",
            "body": f"clips de {P.clip_len_s:g} s cada {P.clip_hop_s:g} s",
            "foot": "`window_starts`",
        },
        {
            "title": "Mel de potencia",
            "body": "el mismo transform\nque en entrenamiento",
            "foot": f"(1, {P.n_mels}, {P.n_frames})",
        },
        {
            "title": "Modelo",
            "body": f"{ARCH['n_queries']} candidatos\npor ventana",
            "foot": "en lotes de 16",
        },
        {
            "title": "Posproceso",
            "body": f"score ≥ {CONFIG.score_threshold:g}\nNMS IoU {CONFIG.nms_iou:g} + IoMin 0,8",
            "foot": "`suppress_nested`",
        },
        {
            "title": "Tabla de Raven",
            "body": "tiempo en s y\nfrecuencia en Hz",
            "foot": "`.selections.txt`",
        },
    ],
    cy,
    margin=2.0,
    gap=2.0,
    wrap=20,
    fill=SURFACE,
    edge=SERIES[2],
    arrow_color=SERIES[2],
    lw=1.3,
    fs=8.4,
    body_fs=7.0,
    foot_fs=6.7,
)
save(fig, "inference_pipeline")
plt.show()

In [ ]:
MAX_EVAL_WINDOWS = 384  # None = la partición entera (obligatorio para las cifras finales)
EVAL_SPLIT = "test"
EVAL_BATCH = 8

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
RUN_MODEL = CHECKPOINT is not None and (CACHE_DIR / f"{EVAL_SPLIT}.pt").exists()
if not RUN_MODEL:
    print("sin checkpoint o sin caché: la sección de resultados no se regenera")

In [ ]:
if RUN_MODEL:
    from torch.utils.data import DataLoader, Subset  # noqa: E402
    from torchvision.ops import box_convert, box_iou  # noqa: E402

    from data.datasets import SpectrogramDataset, collate_fn  # noqa: E402
    from evaluation.metrics import Boxes, concat, hits, overlaps  # noqa: E402
    from models.criterion import HungarianMatcher  # noqa: E402
    from models.registry import load_checkpoint  # noqa: E402
    from utils.boxes import suppress_nested  # noqa: E402

    loaded = load_checkpoint(CHECKPOINT, DEVICE)
    label_names = loaded.labels.names
    dataset = SpectrogramDataset(CACHE_DIR / f"{EVAL_SPLIT}.pt")
    if MAX_EVAL_WINDOWS and MAX_EVAL_WINDOWS < len(dataset):
        index = np.random.default_rng(42).choice(len(dataset), MAX_EVAL_WINDOWS, replace=False)
        subset = Subset(dataset, sorted(index.tolist()))
    else:
        subset = dataset
    loader = DataLoader(subset, batch_size=EVAL_BATCH, shuffle=False, collate_fn=collate_fn)

In [ ]:
if RUN_MODEL:
    from tqdm.auto import tqdm  # noqa: E402

    from models.deformable_detr import predict_scores  # noqa: E402

    @torch.no_grad()
    def collect(model, loader, matcher, n_classes, nms_iou, device, keep_windows=48):
        model.eval()
        predicted_chunks, truth_chunks, windows = [], [], []
        matched_iou, true_classes, predicted_classes = [], [], []
        image_id = 0
        for images, targets in tqdm(loader, desc="inferencia", unit="lote"):
            images = images.to(device)
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
            outputs = model(images)
            boxes = outputs["pred_boxes"]
            scores, labels = predict_scores(outputs)
            decided = outputs["pred_logits"].argmax(-1)

            for b, (query_idx, target_idx) in enumerate(matcher(outputs, targets)):
                target = targets[b]
                if len(query_idx):
                    matched_iou.append(
                        box_iou(
                            box_convert(boxes[b, query_idx], "cxcywh", "xyxy"),
                            box_convert(target["boxes"][target_idx], "cxcywh", "xyxy"),
                        )
                        .diagonal()
                        .cpu()
                    )
                    true_classes.append(target["labels"][target_idx].cpu())
                    predicted_classes.append(decided[b, query_idx].cpu())

                keep = suppress_nested(
                    box_convert(boxes[b], "cxcywh", "xyxy"), scores[b], labels[b], nms_iou
                )
                kept = boxes[b][keep]
                predicted_chunks.append(
                    Boxes(
                        kept,
                        torch.full((len(kept),), image_id, device=kept.device),
                        labels[b][keep],
                        scores[b][keep],
                    )
                )
                n_target = len(target["labels"])
                truth_chunks.append(
                    Boxes(
                        target["boxes"],
                        torch.full((n_target,), image_id, device=target["boxes"].device),
                        target["labels"],
                        torch.ones(n_target, device=target["boxes"].device),
                    )
                )
                if len(windows) < keep_windows:
                    windows.append(
                        {
                            "image": images[b, 0].cpu().numpy(),
                            "truth_boxes": target["boxes"].cpu().numpy(),
                            "truth_labels": target["labels"].cpu().numpy(),
                            "boxes": kept.cpu().numpy(),
                            "labels": labels[b][keep].cpu().numpy(),
                            "scores": scores[b][keep].cpu().numpy(),
                        }
                    )
                image_id += 1

        predictions = concat(predicted_chunks)
        predictions = predictions.select(predictions.scores.argsort(descending=True, stable=True))
        truth = concat(truth_chunks)
        confusion = torch.zeros(n_classes, n_classes + 1, dtype=torch.int64)
        if true_classes:
            confusion.index_put_(
                (torch.cat(true_classes), torch.cat(predicted_classes)),
                torch.ones(len(torch.cat(true_classes)), dtype=torch.int64),
                accumulate=True,
            )
        return {
            "predictions": predictions,
            "truth": truth,
            "matched_iou": torch.cat(matched_iou).numpy() if matched_iou else np.zeros(0),
            "confusion": confusion,
            "windows": windows,
            "n_windows": image_id,
        }

    matcher = HungarianMatcher()
    result = collect(loaded.model, loader, matcher, len(label_names), loaded.nms_iou, DEVICE)

In [ ]:
if RUN_MODEL:
    predictions, truth = result["predictions"], result["truth"]
    n_gt = len(truth.boxes)
    n_pred = len(predictions.boxes)
    iou_threshold = float(loaded.config.get("metric_iou_threshold", 0.5))

    hits_agnostic = hits(overlaps(predictions, truth), n_pred, iou_threshold)
    hits_by_class = hits(overlaps(predictions, truth, class_aware=True), n_pred, iou_threshold)
    scores = predictions.scores.numpy()

    def metrics_at(threshold, hits=None):
        hits = hits_agnostic if hits is None else hits
        k = int((scores >= threshold).sum())
        tp = float(hits[:k].sum())
        recall = tp / n_gt if n_gt else float("nan")
        precision = tp / k if k else float("nan")
        return {
            "k": k,
            "tp": tp,
            "recall": recall,
            "precision": precision,
            "fp_per_tp": (k - tp) / tp if tp else float("nan"),
        }

    operating = loaded.score_threshold
    grid = np.linspace(0.05, 0.95, 37)
    sweep = pd.DataFrame([metrics_at(t) for t in grid], index=grid)
    point = metrics_at(operating)

    fig, axes = plt.subplots(
        2,
        1,
        figsize=(9.4, 6.6),
        sharex=True,
        gridspec_kw={"height_ratios": [1.35, 1], "hspace": 0.42},
    )

    ax = axes[0]
    for column, color, name in (("recall", BLUE, "recall"), ("precision", SERIES[1], "precisión")):
        ax.plot(sweep.index, sweep[column], color=color, linewidth=2.2)
        ax.text(
            sweep.index[-1] + 0.012,
            sweep[column].iloc[-1],
            name,
            color=color,
            fontsize=9,
            va="center",
            fontweight="bold",
        )
    ax.axvline(operating, color=MUTED, linewidth=1.1, linestyle=(0, (4, 3)))
    ax.scatter(
        [operating, operating],
        [point["recall"], point["precision"]],
        s=34,
        color=[BLUE, SERIES[1]],
        zorder=5,
    )
    ax.annotate(
        f"recall {point['recall']:.2f}",
        xy=(operating, point["recall"]),
        xytext=(operating - 0.30, min(point["recall"] + 0.16, 0.98)),
        color=BLUE,
        fontsize=8.5,
        arrowprops=dict(arrowstyle="-", color=BLUE, linewidth=0.9),
    )
    ax.set_ylim(0, 1.02)
    ax.set_xlim(0.03, 1.06)
    ax.set_ylabel("proporción")
    style_value_axis(ax, "y")

    ax = axes[1]
    ax.plot(sweep.index, sweep.fp_per_tp, color=SERIES[2], linewidth=2.2)
    ax.axvline(operating, color=MUTED, linewidth=1.1, linestyle=(0, (4, 3)))
    ax.scatter([operating], [point["fp_per_tp"]], s=34, color=SERIES[2], zorder=5)
    ax.annotate(
        f"{point['fp_per_tp']:.2f} falsos positivos por acierto",
        xy=(operating, point["fp_per_tp"]),
        xytext=(operating + 0.06, point["fp_per_tp"] * 1.35),
        color=SERIES[2],
        fontsize=8.5,
        arrowprops=dict(arrowstyle="-", color=SERIES[2], linewidth=0.9),
    )
    ax.set_ylim(0, min(np.nanmax(sweep.fp_per_tp) * 1.15, 12))
    ax.set_xlabel("umbral de score")
    ax.set_ylabel("FP / TP")
    style_value_axis(ax, "y")

    save(
        fig,
        "score_sweep",
        data={
            "umbral": sweep.index,
            "recall": sweep.recall,
            "precision": sweep.precision,
            "fp_por_tp": sweep.fp_per_tp,
            "operacion": {"umbral": operating, **point},
        },
    )
    plt.show()

In [ ]:
if RUN_MODEL:
    unreachable = (
        experiment.assign(long=experiment.duration_s > 2 * P.clip_len_s).groupby("pair").long.mean()
    )
    bandwidth_cv = (
        experiment.groupby("pair").bandwidth_hz.std()
        / experiment.groupby("pair").bandwidth_hz.mean()
    )
    median_ms = experiment.groupby("pair").duration_s.median() * 1000
    nested_children = {syllable for _, syllable in NESTED_PAIRS}

    def attribute(pair):
        if unreachable.get(pair, 0) > 0.2:
            return f"{unreachable[pair]:.0%} inalcanzable (> {2 * P.clip_len_s:g} s)"
        if pair in nested_children:
            return "anidada en una clase de frase excluida"
        if bandwidth_cv.get(pair, 0) > 0.45:
            return f"geometría heterogénea (CV de banda {bandwidth_cv[pair]:.2f})"
        if median_ms.get(pair, 1e9) < 120:
            return f"mediana de {median_ms[pair]:.0f} ms"
        return ""

    rows = []
    predicted_labels = predictions.labels.numpy()
    k = int((scores >= operating).sum())
    for class_id, name in enumerate(label_names):
        class_gt = int((truth.labels == class_id).sum())
        if not class_gt:
            continue
        tp = float(hits_by_class[:k][predicted_labels[:k] == class_id].sum())
        rows.append(
            {
                "clase": name,
                "recall": tp / class_gt,
                "cajas": class_gt,
                "nota": attribute(name),
            }
        )
    per_class = pd.DataFrame(rows).sort_values("recall", ascending=False).reset_index(drop=True)

    fig, ax = plt.subplots(figsize=(10.6, 0.42 * len(per_class) + 2.2))
    ys = np.arange(len(per_class))
    ax.set_xlim(0, 1.72)
    ax.set_ylim(len(per_class) - 0.4, -0.9)
    barh(ax, ys, per_class.recall.values, BLUE, height=0.6)
    for y, row in per_class.iterrows():
        ax.text(
            row.recall + 0.012,
            y,
            f"{row.recall:.2f}",
            va="center",
            fontsize=8,
            color=INK,
            fontweight="bold",
        )
        if row.nota:
            ax.text(1.06, y, row.nota, va="center", fontsize=7.6, color=INK_2)
    ax.axvline(1.0, color=BASELINE, linewidth=1.0)
    ax.set_yticks(
        ys, [f"{row.clase}  ({row.cajas:,})" for _, row in per_class.iterrows()], fontsize=8
    )
    ax.set_xticks(np.arange(0, 1.01, 0.25))
    style_value_axis(ax, "x")
    ax.set_xlabel("recall por clase al punto de operación")
    save(
        fig,
        "recall_per_class",
        data=per_class,
    )
    plt.show()

In [ ]:
if RUN_MODEL:
    confusion = result["confusion"].numpy()
    totals = confusion.sum(axis=1, keepdims=True)
    normalized = np.divide(confusion, np.maximum(totals, 1))
    present = totals[:, 0] > 0
    names_present = [n for n, keep in zip(label_names, present) if keep]
    matrix = normalized[present]

    fig, ax = plt.subplots(
        figsize=(0.46 * (len(label_names) + 2) + 3.0, 0.46 * len(names_present) + 2.6)
    )
    ax.imshow(matrix[:, :-1], cmap="Blues", vmin=0, vmax=1, aspect="auto")
    ax.imshow(
        matrix[:, -1:],
        cmap="Oranges",
        vmin=0,
        vmax=1,
        aspect="auto",
        extent=(
            len(label_names) + 0.1 - 0.5,
            len(label_names) + 1.1 - 0.5,
            len(names_present) - 0.5,
            -0.5,
        ),
    )
    for i in range(matrix.shape[0]):
        for j in range(matrix.shape[1]):
            value = matrix[i, j]
            if value < 0.005:
                continue
            x = j if j < matrix.shape[1] - 1 else len(label_names) + 0.6 - 0.5
            ax.text(
                x,
                i,
                f"{value:.0%}",
                ha="center",
                va="center",
                fontsize=6.8,
                color=SURFACE if value > 0.55 else INK_2,
            )
    ax.set_xlim(-0.6, len(label_names) + 1.2)
    ax.set_xticks(
        list(range(len(label_names))) + [len(label_names) + 0.6 - 0.5],
        label_names + ["∅"],
        rotation=90,
        fontsize=7.5,
    )
    ax.set_yticks(range(len(names_present)), names_present, fontsize=7.5)
    ax.set_xlabel("clase predicha")
    ax.set_ylabel("clase anotada")
    for side in ("top", "right", "left", "bottom"):
        ax.spines[side].set_visible(False)
    save(
        fig,
        "confusion_matrix",
        data={
            "filas": names_present,
            "columnas": [*label_names, "sin detección"],
            "matriz_normalizada": matrix,
            "conteos": confusion[present],
        },
    )
    plt.show()

In [ ]:
if RUN_MODEL:
    values = result["matched_iou"]
    fig, ax = plt.subplots(figsize=(9.4, 4.6))
    counts, edges = np.histogram(values, bins=np.linspace(0, 1, 41))
    centres = (edges[:-1] + edges[1:]) / 2
    ax.set_xlim(0, 1)
    ax.set_ylim(0, counts.max() * 1.22)
    barv(ax, centres, counts, BLUE, width=(edges[1] - edges[0]) * 0.88)
    for threshold, color in ((0.25, MUTED), (0.5, SERIES[1]), (0.75, CRITICAL)):
        share = float((values >= threshold).mean())
        ax.axvline(threshold, color=color, linewidth=1.2, linestyle=(0, (4, 3)), zorder=3)
        ax.text(
            threshold + 0.008,
            counts.max() * 1.16,
            f"IoU ≥ {threshold:g}\n{share:.0%} de los pares",
            color=color,
            fontsize=8,
            va="top",
        )
    ax.set_xlabel("IoU entre la caja predicha y la anotada")
    ax.set_ylabel("pares emparejados")
    style_value_axis(ax, "y")
    save(
        fig,
        "iou_distribution",
        data={
            "borde": edges,
            "pares": counts,
            "sobre_umbral": {t: float((values >= t).mean()) for t in (0.25, 0.5, 0.75)},
        },
    )
    plt.show()

In [ ]:
if RUN_MODEL:
    from torchvision.ops import box_iou  # noqa: E402

    def window_stats(window, threshold):
        keep = window["scores"] >= threshold
        predicted = window["boxes"][keep]
        truth_boxes = window["truth_boxes"]
        best = np.zeros(len(truth_boxes))
        best_class = np.zeros(len(truth_boxes), dtype=bool)
        if len(predicted) and len(truth_boxes):
            iou = box_iou(
                box_convert(torch.tensor(predicted), "cxcywh", "xyxy"),
                box_convert(torch.tensor(truth_boxes), "cxcywh", "xyxy"),
            ).numpy()
            best = iou.max(axis=0) if len(predicted) else best
            argmax = iou.argmax(axis=0)
            best_class = window["labels"][keep][argmax] == window["truth_labels"]
        unmatched = []
        if len(predicted):
            iou_pred = (
                box_iou(
                    box_convert(torch.tensor(predicted), "cxcywh", "xyxy"),
                    box_convert(torch.tensor(truth_boxes), "cxcywh", "xyxy"),
                )
                .numpy()
                .max(axis=1)
                if len(truth_boxes)
                else np.zeros(len(predicted))
            )
            unmatched = window["scores"][keep][iou_pred < 0.25]
        return {
            "n_truth": len(truth_boxes),
            "n_pred": int(keep.sum()),
            "best_iou": best,
            "class_ok": best_class,
            "top_false": float(unmatched.max()) if len(unmatched) else 0.0,
        }

    stats = [window_stats(w, operating) for w in result["windows"]]

    def choose(predicate, key, default_key=None):
        candidates = [(i, s) for i, s in enumerate(stats) if predicate(s)]
        if not candidates:
            return None
        return max(candidates, key=lambda pair: key(pair[1]))[0]

    picks = [
        (
            "Acierto limpio",
            choose(
                lambda s: (
                    s["n_truth"] >= 1
                    and len(s["best_iou"])
                    and s["best_iou"].min() >= 0.6
                    and s["class_ok"].all()
                ),
                lambda s: s["n_truth"],
            ),
        ),
        (
            "Escena densa",
            choose(
                lambda s: s["n_truth"] >= 3, lambda s: (s["n_truth"], (s["best_iou"] >= 0.5).sum())
            ),
        ),
        (
            "Encuadre flojo",
            choose(
                lambda s: (
                    s["n_truth"] >= 1 and len(s["best_iou"]) and 0.05 < s["best_iou"].min() < 0.5
                ),
                lambda s: -s["best_iou"].min(),
            ),
        ),
        (
            "Propuesta de más",
            choose(lambda s: s["top_false"] > 0.6, lambda s: s["top_false"]),
        ),
    ]
    picks = [(name, index) for name, index in picks if index is not None]

    fig, axes = plt.subplots(
        2, 2, figsize=(13.0, 7.2), gridspec_kw={"hspace": 0.42, "wspace": 0.16}
    )
    for ax, (name, index) in zip(axes.ravel(), picks):
        window = result["windows"][index]
        stat = stats[index]
        draw_spectrogram(ax, np.log(window["image"] + P.eps), P.clip_len_s)
        for box, label in zip(window["truth_boxes"], window["truth_labels"]):
            cx, cy, w, h = box
            ax.add_patch(
                Rectangle(
                    ((cx - w / 2) * P.clip_len_s, cy - h / 2),
                    w * P.clip_len_s,
                    h,
                    facecolor="none",
                    edgecolor=GOOD,
                    linewidth=1.6,
                    linestyle=(0, (4, 2)),
                    zorder=4,
                )
            )
            ax.text(
                (cx - w / 2) * P.clip_len_s,
                cy + h / 2 + 0.012,
                label_names[int(label)],
                color=GOOD,
                fontsize=7,
                fontweight="bold",
                va="bottom",
            )
        keep = window["scores"] >= operating
        for box, label, score in zip(
            window["boxes"][keep], window["labels"][keep], window["scores"][keep]
        ):
            cx, cy, w, h = box
            ax.add_patch(
                Rectangle(
                    ((cx - w / 2) * P.clip_len_s, cy - h / 2),
                    w * P.clip_len_s,
                    h,
                    facecolor="none",
                    edgecolor=BLUE,
                    linewidth=1.5,
                    zorder=5,
                )
            )
            ax.text(
                (cx - w / 2) * P.clip_len_s,
                cy - h / 2 - 0.018,
                f"{label_names[int(label)]} {score:.2f}",
                color=BLUE,
                fontsize=7,
                va="top",
            )
        detail = (
            f"{stat['n_truth']} anotada{'s' if stat['n_truth'] != 1 else ''} · "
            f"{stat['n_pred']} propuesta{'s' if stat['n_pred'] != 1 else ''}"
            + (f" · IoU mín {stat['best_iou'].min():.2f}" if len(stat["best_iou"]) else "")
        )
        ax.set_title(name, fontsize=10.5, pad=22)
        ax.annotate(
            detail,
            xy=(1, 1),
            xytext=(0, 5),
            xycoords="axes fraction",
            textcoords="offset points",
            ha="right",
            va="bottom",
            fontsize=8,
            color=INK_2,
        )
        ax.set_xlabel("")
        ax.set_ylabel("")

    handles = [
        plt.Line2D([], [], color=GOOD, linewidth=2, linestyle=(0, (4, 2)), label="anotación"),
        plt.Line2D([], [], color=BLUE, linewidth=2, label="predicción sobre el umbral"),
    ]
    fig.legend(handles=handles, loc="lower left", bbox_to_anchor=(0.005, -0.02), ncol=2)
    save(fig, "qualitative_detections")
    plt.show()

In [ ]:
ABLATION = {
    "Deformable DETR base": {"AP@0.25": 0.609, "AP@0.50": 0.325, "AP@0.75": 0.037, "épocas": 30},
    "+ init de atención, refinamiento\niterativo y pérdidas auxiliares": {
        "AP@0.25": 0.742,
        "AP@0.50": 0.569,
        "AP@0.75": 0.112,
        "épocas": 75,
    },
    "+ IoU en el matcher y en la pérdida": {
        "AP@0.25": 0.803,
        "AP@0.50": 0.648,
        "AP@0.75": 0.133,
        "épocas": 20,
    },
}
ABLATION_SOURCE = "reporte de avance, subconjunto de 10 clases"

table = pd.DataFrame(ABLATION).T
metrics = ["AP@0.25", "AP@0.50", "AP@0.75"]
fig, ax = plt.subplots(figsize=(10.0, 4.8))
xs = np.arange(len(table))
step = 0.26
ax.set_ylim(0, table[metrics].values.max() * 1.28)
ax.set_xlim(-0.6, len(table) - 0.4)
for i, (metric, color) in enumerate(zip(metrics, SERIES)):
    barv(ax, xs + (i - 1) * step, table[metric].values, color, width=step * 0.82)
    for x, value in zip(xs + (i - 1) * step, table[metric].values):
        ax.text(
            x, value + 0.015, f"{value:.3f}", ha="center", va="bottom", fontsize=7.6, color=INK_2
        )
ax.set_xticks(
    xs, [f"{name}\n({int(row['épocas'])} épocas)" for name, row in table.iterrows()], fontsize=8.5
)
ax.set_ylabel("AP agnóstico de clase")
style_value_axis(ax, "y")
handles = [
    plt.Line2D([], [], marker="s", linestyle="", markersize=8, color=c, label=m)
    for c, m in zip(SERIES, metrics)
]
ax.legend(handles=handles, loc="upper left", ncol=3, bbox_to_anchor=(0, 1.02))
save(
    fig,
    "ablation_progression",
    data={"variantes": ABLATION, "fuente": ABLATION_SOURCE},
)
plt.show()

In [ ]:
if RUN_MODEL and splits:
    from inference.predictor import predict  # noqa: E402

    durations = recordings.set_index("audio_path").duration_s
    test_files = {w.audio_path for w in splits["test"]}
    counts = experiment[experiment.audio_path.isin(test_files)].audio_path.value_counts()
    candidates = [
        (path, n) for path, n in counts.items() if 4 <= n <= 14 and durations.get(path, 1e9) <= 90
    ]
    if not candidates:
        candidates = [(counts.index[0], counts.iloc[0])]
    recording_path, _ = min(candidates, key=lambda pair: abs(pair[1] - 8))
    recording_duration = float(durations.get(recording_path, 30.0))

    # `predict` recibe el `LoadedModel` entero: desde el registro, las etiquetas y el
    # punto de operación viajan con el modelo en vez de pasarse sueltos.
    table = predict(
        loaded,
        recording_path,
        DEVICE,
        score_threshold=loaded.score_threshold,
        nms_iou=loaded.nms_iou,
        batch_size=8,
    )

In [ ]:
if RUN_MODEL and splits:
    span = min(recording_duration, 15.0)
    waveform = load_segment(recording_path, 0.0, span)
    truth_rows = experiment[
        (experiment.audio_path == recording_path) & (experiment.begin_time_s < span)
    ]

    fig, ax = plt.subplots(figsize=(13.0, 5.0))
    draw_spectrogram(ax, log_mel(waveform), span)
    for r in truth_rows.itertuples():
        draw_box(
            ax,
            r.begin_time_s,
            min(r.end_time_s, span),
            r.low_freq_hz,
            r.high_freq_hz,
            color=GOOD,
            lw=1.5,
            dashed=True,
        )
    shown = table[table["Begin Time (s)"] < span]
    labelled = set(shown.nlargest(min(5, len(shown)), "Score").index)
    for index, r in shown.iterrows():
        draw_box(
            ax,
            r["Begin Time (s)"],
            min(r["End Time (s)"], span),
            r["Low Freq (Hz)"],
            r["High Freq (Hz)"],
            color=BLUE,
            lw=1.4,
            label=(
                f"{r['Species']}/{r['Call type']} {r['Score']:.2f}" if index in labelled else None
            ),
            fs=6.5,
            label_below=True,
        )
    handles = [
        plt.Line2D(
            [],
            [],
            color=GOOD,
            linewidth=2,
            linestyle=(0, (4, 2)),
            label=f"anotación del experto ({len(truth_rows)})",
        ),
        plt.Line2D([], [], color=BLUE, linewidth=2, label=f"detección del modelo ({len(shown)})"),
    ]
    ax.legend(handles=handles, loc="lower left", bbox_to_anchor=(0, -0.30), ncol=2)
    save(fig, "detection_timeline")
    plt.show()